# Local search (C++)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/combinatorial-optimization/blob/main/notebooks/local-search_cpp.ipynb)

Executable notebook of the chapter “Local search” (part “Heuristic algorithms” of the course *Combinatorial Optimization*): it contains the local search of the chapter for the knapsack problem, the bin packing problem, the TSP and $P\,||\,C_{\max}$: the generic scheme, variable neighbourhood descent, the steps of every neighbourhood and the multi-start 2-opt, written in C++ and run on every example of the chapter and on every instance of the exercises, checking each result against the expected one.

The full chapter — pseudo-code, examples, proofs — is [on the website](https://fabiofurini.github.io/combinatorial-optimization/heuristic-algorithms/local-search/). The same notebook is also available in [Python](https://colab.research.google.com/github/fabiofurini/combinatorial-optimization/blob/main/notebooks/local-search_python.ipynb).

**How to run it.** On Colab click the badge and then *Runtime → Run all*, or run the cells one at a time with `Shift+Enter`. Each C++ file is written to the folder `src/` by its `%%writefile` cell; the cell after `Main.cpp` compiles everything with `g++ -std=c++17 -O2` and runs the examples. Nothing to install (Colab already has `g++`).

**Indices.** In the notes items, jobs, cities, machines and bins are numbered from 1, here from 0: the algorithmic logic is unchanged.

## Declarations

The header declares all the functions of the chapter; each function is in its own file, with the same name.

In [1]:
%%writefile src/LocalSearch.h
#ifndef LOCAL_SEARCH_HEADER
#define LOCAL_SEARCH_HEADER

// local search for the KP, the BPP, the TSP and P||Cmax
// items, jobs, cities, machines and bins are indexed 0, 1, ...
// every Step function looks for an improving neighbour of the current solution:
// if it finds one, it applies the move to the solution and returns true; otherwise it returns false
// rule = FIRST (first improvement) or BEST (best improvement)

#include <vector>
#include <functional>

enum { FIRST = 0, BEST = 1 };
enum { CMAX = 0, SQ = 1, NB = 2 };     // objective: makespan, sum of squared loads, number of bins

typedef std::vector<int> Vec;
typedef std::vector<std::vector<int>> Mat;

// generic schemes
int LocalSearch(const std::function<bool()>& Step);
int VND(const std::vector<std::function<bool()>>& Steps);

// KP
bool KPFlipSwapStep(int n, const Vec& p, const Vec& w, int c, Vec& x, int rule);
bool KPPenaltyFlipStep(int n, const Vec& p, const Vec& w, int c, int alpha_num, int alpha_den, Vec& x, int rule);

// BPP
bool BPPMoveStep(int n, const Vec& w, int c, Vec& bin, Vec& L, int obj, int rule);
bool BPPSwapStep(int n, const Vec& w, int c, Vec& bin, Vec& L, int rule);

// TSP
bool TwoOptStep(int n, const Mat& d, Vec& pi, int rule);
bool OrOptStep(int n, const Mat& d, Vec& pi, int rule);
int MultiStartTwoOpt(int n, const Mat& d, int M, long long seed, int rule, Vec& best);

// P||Cmax
int ObjectiveChange(int m, const Vec& L, int a, int b, int delta, int obj);
bool PCmaxMoveStep(int n, int m, const Vec& t, Vec& mach, Vec& L, int obj, int rule);
bool PCmaxSwapStep(int n, int m, const Vec& t, Vec& mach, Vec& L, int obj, int rule);

// starting solutions, random numbers and evaluation (recalled from the notes on constructive heuristics)
double ParkMiller(long long& seed);
int RandomInt(long long& seed, int k);
Vec RandomTour(int n, long long& seed);
Vec RandomPermutation(int n, long long& seed);
Vec RandomAssignment(int n, int m, long long& seed);
int TourLength(int n, const Mat& d, const Vec& pi);
Vec NearestNeighbour(int n, const Mat& d, int s);
Vec LPT(int n, int m, const Vec& t);
Vec FirstFit(int n, const Vec& w, int c, const Vec& order);
Vec NextFit(int n, const Vec& w, int c);
Vec GreedyKP(int n, const Vec& p, const Vec& w, int c, const Vec& order);
Vec Loads(int n, int k, const Vec& assign, const Vec& size);
int NumberOfBins(int n, const Vec& bin);
Mat EuclideanDistances(const std::vector<std::pair<int,int>>& pts);

#endif

Writing src/LocalSearch.h


## Starting solutions, random numbers and evaluation

The starting solutions (recalled from the chapter on constructive heuristics), the Park–Miller generator, the random solutions and the evaluation functions. The constants `FIRST`, `BEST` select the rule, `CMAX`, `SQ`, `NB` the objective.

In [2]:
%%writefile src/ParkMiller.cpp
#include "LocalSearch.h"

// Park-Miller minimal standard generator: seed = 16807 * seed mod (2^31 - 1)
// returns u = seed / (2^31 - 1), a pseudo-random number in (0, 1)

double ParkMiller(long long& seed)
{
    seed = (16807LL * seed) % 2147483647LL;
    return (double)seed / 2147483647.0;
}

Writing src/ParkMiller.cpp


In [3]:
%%writefile src/RandomInt.cpp
#include "LocalSearch.h"

// pseudo-random integer in {0, 1, ..., k-1}: floor(u * k), computed in exact integer arithmetic

int RandomInt(long long& seed, int k)
{
    ParkMiller(seed);
    return (int)((seed * (long long)k) / 2147483647LL);
}

Writing src/RandomInt.cpp


In [4]:
%%writefile src/RandomTour.cpp
#include "LocalSearch.h"

// random tour with city 0 in the first position (Fisher-Yates shuffle of the positions 1..n-1)

Vec RandomTour(int n, long long& seed)
{
    Vec pi(n);
    for (int i = 0; i < n; ++i) pi[i] = i;
    for (int i = n - 1; i >= 2; --i)
    {
        int j = 1 + RandomInt(seed, i);                 // j in {1, ..., i}
        int tmp = pi[i]; pi[i] = pi[j]; pi[j] = tmp;
    }
    return pi;
}

Writing src/RandomTour.cpp


In [5]:
%%writefile src/RandomPermutation.cpp
#include "LocalSearch.h"

// random permutation of 0..n-1 (Fisher-Yates shuffle)

Vec RandomPermutation(int n, long long& seed)
{
    Vec a(n);
    for (int i = 0; i < n; ++i) a[i] = i;
    for (int i = n - 1; i >= 1; --i)
    {
        int j = RandomInt(seed, i + 1);                 // j in {0, ..., i}
        int tmp = a[i]; a[i] = a[j]; a[j] = tmp;
    }
    return a;
}

Writing src/RandomPermutation.cpp


In [6]:
%%writefile src/RandomAssignment.cpp
#include "LocalSearch.h"

// random assignment of n jobs to m machines

Vec RandomAssignment(int n, int m, long long& seed)
{
    Vec mach(n);
    for (int j = 0; j < n; ++j) mach[j] = RandomInt(seed, m);
    return mach;
}

Writing src/RandomAssignment.cpp


In [7]:
%%writefile src/TourLength.cpp
#include "LocalSearch.h"

// length of the tour pi[0..n-1]

int TourLength(int n, const Mat& d, const Vec& pi)
{
    int length = 0;
    for (int k = 0; k < n; ++k) length = length + d[pi[k]][pi[(k + 1) % n]];
    return length;
}

Writing src/TourLength.cpp


In [8]:
%%writefile src/NearestNeighbour.cpp
#include "LocalSearch.h"

// nearest neighbour heuristic from city s (ties broken by the smallest index);
// the tour is returned rotated so that it starts from city 0

Vec NearestNeighbour(int n, const Mat& d, int s)
{
    Vec tour(1, s);
    std::vector<bool> visited(n, false);
    visited[s] = true;
    while ((int)tour.size() < n)
    {
        int last = tour.back(), next = -1;
        for (int j = 0; j < n; ++j)
            if (!visited[j] && (next == -1 || d[last][j] < d[last][next])) next = j;
        tour.push_back(next); visited[next] = true;
    }
    Vec pi;
    int k0 = 0;
    while (tour[k0] != 0) ++k0;
    for (int k = 0; k < n; ++k) pi.push_back(tour[(k0 + k) % n]);
    return pi;
}

Writing src/NearestNeighbour.cpp


In [9]:
%%writefile src/LPT.cpp
#include "LocalSearch.h"

// longest processing time first: the jobs, by non-increasing time (ties by index),
// are assigned to the least loaded machine (ties by index)

Vec LPT(int n, int m, const Vec& t)
{
    Vec order(n);
    for (int j = 0; j < n; ++j) order[j] = j;
    for (int a = 0; a < n; ++a)                         // insertion sort, stable
        for (int b = a; b > 0 && t[order[b]] > t[order[b - 1]]; --b)
        { int tmp = order[b]; order[b] = order[b - 1]; order[b - 1] = tmp; }
    Vec L(m, 0), mach(n, 0);
    for (int a = 0; a < n; ++a)
    {
        int j = order[a], h = 0;
        for (int q = 1; q < m; ++q) if (L[q] < L[h]) h = q;
        mach[j] = h; L[h] = L[h] + t[j];
    }
    return mach;
}

Writing src/LPT.cpp


In [10]:
%%writefile src/FirstFit.cpp
#include "LocalSearch.h"

// first fit, with the items examined in the given order; bins are labelled 0, 1, ... in opening order

Vec FirstFit(int n, const Vec& w, int c, const Vec& order)
{
    Vec L, bin(n, 0);
    for (int a = 0; a < n; ++a)
    {
        int i = order[a], b = 0;
        while (b < (int)L.size() && L[b] + w[i] > c) ++b;
        if (b == (int)L.size()) L.push_back(0);         // open a new bin
        L[b] = L[b] + w[i]; bin[i] = b;
    }
    return bin;
}

Writing src/FirstFit.cpp


In [11]:
%%writefile src/NextFit.cpp
#include "LocalSearch.h"

// next fit: only the last opened bin is considered

Vec NextFit(int n, const Vec& w, int c)
{
    Vec bin(n, 0);
    int b = 0, load = 0;
    for (int i = 0; i < n; ++i)
    {
        if (load + w[i] > c) { b = b + 1; load = 0; }   // close the bin, open a new one
        load = load + w[i]; bin[i] = b;
    }
    return bin;
}

Writing src/NextFit.cpp


In [12]:
%%writefile src/GreedyKP.cpp
#include "LocalSearch.h"

// greedy algorithm for the KP, with the items examined in the given order

Vec GreedyKP(int n, const Vec& p, const Vec& w, int c, const Vec& order)
{
    (void)p;
    Vec x(n, 0);
    int rc = c;                                         // residual capacity
    for (int a = 0; a < n; ++a)
    {
        int i = order[a];
        if (w[i] <= rc) { x[i] = 1; rc = rc - w[i]; }
    }
    return x;
}

Writing src/GreedyKP.cpp


In [13]:
%%writefile src/Loads.cpp
#include "LocalSearch.h"

// loads of k bins (or machines): L[b] = sum of size[i] over the i with assign[i] = b

Vec Loads(int n, int k, const Vec& assign, const Vec& size)
{
    Vec L(k, 0);
    for (int i = 0; i < n; ++i) L[assign[i]] = L[assign[i]] + size[i];
    return L;
}

Writing src/Loads.cpp


In [14]:
%%writefile src/NumberOfBins.cpp
#include "LocalSearch.h"

// number of non-empty bins

int NumberOfBins(int n, const Vec& bin)
{
    std::vector<bool> used(n, false);
    int nb = 0;
    for (int i = 0; i < n; ++i)
        if (!used[bin[i]]) { used[bin[i]] = true; nb = nb + 1; }
    return nb;
}

Writing src/NumberOfBins.cpp


In [15]:
%%writefile src/EuclideanDistances.cpp
#include "LocalSearch.h"
#include <cmath>

// Euclidean distances between points with integer coordinates, rounded to the nearest integer

Mat EuclideanDistances(const std::vector<std::pair<int,int>>& pts)
{
    int n = (int)pts.size();
    Mat d(n, Vec(n, 0));
    for (int i = 0; i < n; ++i)
        for (int j = 0; j < n; ++j)
        {
            double dx = pts[i].first - pts[j].first, dy = pts[i].second - pts[j].second;
            d[i][j] = (int)std::llround(std::sqrt(dx * dx + dy * dy));
        }
    return d;
}

Writing src/EuclideanDistances.cpp


## Local search and variable neighbourhood descent

The generic **local search** and the **variable neighbourhood descent** receive the steps as functions without arguments, which act on the current solution. Every step looks for an improving neighbour: if it finds one, it applies the move and returns true.

In [16]:
%%writefile src/LocalSearch.cpp
#include "LocalSearch.h"

// generic local search: Step looks for an improving neighbour of the current solution
// and, if it finds one, moves to it and returns true
// the function returns the number of moves performed

int LocalSearch(const std::function<bool()>& Step)
{
    int moves = 0;                      // number of moves performed
    while (Step())                      // an improving neighbour was found and accepted
    {
        moves = moves + 1;
    }
    return moves;                       // the current solution is a local optimum
}

Writing src/LocalSearch.cpp


In [17]:
%%writefile src/VND.cpp
#include "LocalSearch.h"

// variable neighbourhood descent: Steps[0], ..., Steps[K-1] explore the neighbourhoods N_1, ..., N_K
// after every improving move the search restarts from the first neighbourhood
// the function returns the number of moves performed

int VND(const std::vector<std::function<bool()>>& Steps)
{
    int K = (int)Steps.size();          // number of neighbourhoods
    int k = 0;                          // current neighbourhood
    int moves = 0;                      // number of moves performed
    while (k < K)
    {
        if (Steps[k]())                 // improving neighbour found in N_k, and accepted
        {
            moves = moves + 1;
            k = 0;                      // back to the first neighbourhood
        }
        else
        {
            k = k + 1;                  // local optimum for N_k: try the next neighbourhood
        }
    }
    return moves;                       // local optimum for all the K neighbourhoods
}

Writing src/VND.cpp


## TSP: 2-opt and Or-opt

The **2-opt step** and the **Or-opt step**; city $0$ stays in the first position of the tour.

In [18]:
%%writefile src/TwoOptStep.cpp
#include "LocalSearch.h"

// one step of the 2-opt local search on the tour pi[0..n-1] (pi[0] never moves)
// the move (i, j) removes the edges {pi[i], pi[i+1]} and {pi[j], pi[j+1]},
// reverses the path pi[i+1..j] and adds the edges {pi[i], pi[j]} and {pi[i+1], pi[j+1]}

bool TwoOptStep(int n, const Mat& d, Vec& pi, int rule)
{
    int bestDelta = 0;                  // best length change found so far
    int bi = -1, bj = -1;               // best move found so far
    bool stop = false;
    for (int i = 0; i <= n - 3 && !stop; ++i)
    {
        for (int j = i + 2; j <= n - 1; ++j)
        {
            if (i == 0 && j == n - 1) continue;          // the two edges are adjacent
            int u1 = pi[i], v1 = pi[i + 1];               // first removed edge
            int u2 = pi[j], v2 = pi[(j + 1) % n];         // second removed edge
            int delta = d[u1][u2] + d[v1][v2] - d[u1][v1] - d[u2][v2];
            if (delta < bestDelta)
            {
                bestDelta = delta; bi = i; bj = j;        // improving move
                if (rule == FIRST) { stop = true; break; }
            }
        }
    }
    if (bi == -1) return false;         // no improving move: local optimum

    for (int a = bi + 1, b = bj; a < b; ++a, --b)        // reverse pi[bi+1..bj]
    {
        int tmp = pi[a]; pi[a] = pi[b]; pi[b] = tmp;
    }
    return true;
}

Writing src/TwoOptStep.cpp


In [19]:
%%writefile src/OrOptStep.cpp
#include "LocalSearch.h"

// one step of the Or-opt local search on the tour pi[0..n-1] (pi[0] never moves)
// the move (l, i, k) takes the segment pi[i..i+l-1] of l = 1, 2, 3 consecutive cities
// and inserts it, with the same orientation, between pi[k] and pi[k+1]

bool OrOptStep(int n, const Mat& d, Vec& pi, int rule)
{
    int bestDelta = 0;                  // best length change found so far
    int bl = -1, bi = -1, bk = -1;      // best move found so far
    bool stop = false;
    for (int l = 1; l <= 3 && !stop; ++l)
    {
        for (int i = 1; i + l - 1 <= n - 1 && !stop; ++i)
        {
            int p = pi[i - 1], s1 = pi[i];                // predecessor and first city of the segment
            int sl = pi[i + l - 1], q = pi[(i + l) % n];  // last city of the segment and successor
            for (int k = 0; k <= n - 1; ++k)
            {
                if (k >= i - 1 && k <= i + l - 1) continue;   // same position, or inside the segment
                int a = pi[k], b = pi[(k + 1) % n];           // edge where the segment is inserted
                int delta = d[p][q] + d[a][s1] + d[sl][b] - d[p][s1] - d[sl][q] - d[a][b];
                if (delta < bestDelta)
                {
                    bestDelta = delta; bl = l; bi = i; bk = k;   // improving move
                    if (rule == FIRST) { stop = true; break; }
                }
            }
        }
    }
    if (bl == -1) return false;         // no improving move: local optimum

    Vec newpi;                          // the new tour
    for (int k = 0; k <= n - 1; ++k)
    {
        if (k >= bi && k <= bi + bl - 1) continue;        // the segment is copied after pi[bk]
        newpi.push_back(pi[k]);
        if (k == bk)
            for (int r = bi; r <= bi + bl - 1; ++r) newpi.push_back(pi[r]);
    }
    pi = newpi;
    return true;
}

Writing src/OrOptStep.cpp


## $P\,||\,C_{\max}$: move and swap

The change $F$ of the objective of $P\,||\,C_{\max}$, for the makespan and for the sum of the squared loads, and the **move** and **swap steps**.

In [20]:
%%writefile src/ObjectiveChange.cpp
#include "LocalSearch.h"

// change of the objective of P||Cmax when the time delta is transferred from machine a to machine b
// (a move transfers delta = t_j, a swap transfers delta = t_j - t_k)
// obj = CMAX: change of the makespan; obj = SQ: change of the sum of the squared loads

int ObjectiveChange(int m, const Vec& L, int a, int b, int delta, int obj)
{
    if (obj == SQ)
        return 2 * delta * (L[b] - L[a] + delta);

    int Cmax = 0, newC = 0;                             // makespan before and after
    for (int q = 0; q < m; ++q)
    {
        if (L[q] > Cmax) Cmax = L[q];
        int newL = L[q];
        if (q == a) newL = L[q] - delta;
        if (q == b) newL = L[q] + delta;
        if (newL > newC) newC = newL;
    }
    return newC - Cmax;
}

Writing src/ObjectiveChange.cpp


In [21]:
%%writefile src/PCmaxMoveStep.cpp
#include "LocalSearch.h"

// one step of the move local search for P||Cmax
// mach[j] is the machine of job j, L[h] the load of machine h
// obj = CMAX (makespan) or SQ (sum of the squared loads); both are minimized

bool PCmaxMoveStep(int n, int m, const Vec& t, Vec& mach, Vec& L, int obj, int rule)
{
    int bestDelta = 0;                                  // best change of the objective found so far
    int bj = -1, bh = -1;                               // best move found so far
    bool stop = false;
    for (int j = 0; j < n && !stop; ++j)
    {
        int a = mach[j];                                // machine of job j
        for (int h = 0; h < m; ++h)
        {
            if (h == a) continue;
            int delta = ObjectiveChange(m, L, a, h, t[j], obj);   // job j from a to h
            if (delta < bestDelta)
            {
                bestDelta = delta; bj = j; bh = h;      // improving move
                if (rule == FIRST) { stop = true; break; }
            }
        }
    }
    if (bj == -1) return false;                         // no improving move: local optimum

    L[mach[bj]] = L[mach[bj]] - t[bj];                  // apply the move
    L[bh] = L[bh] + t[bj];
    mach[bj] = bh;
    return true;
}

Writing src/PCmaxMoveStep.cpp


In [22]:
%%writefile src/PCmaxSwapStep.cpp
#include "LocalSearch.h"

// one step of the swap local search for P||Cmax
// the move (j, k) exchanges the machines of the jobs j and k
// obj = CMAX (makespan) or SQ (sum of the squared loads); both are minimized

bool PCmaxSwapStep(int n, int m, const Vec& t, Vec& mach, Vec& L, int obj, int rule)
{
    int bestDelta = 0;                                  // best change of the objective found so far
    int bj = -1, bk = -1;                               // best move found so far
    bool stop = false;
    for (int j = 0; j < n && !stop; ++j)
    {
        for (int k = j + 1; k < n; ++k)
        {
            int a = mach[j], b = mach[k];
            if (a == b || t[j] == t[k]) continue;       // the loads would not change
            int delta = ObjectiveChange(m, L, a, b, t[j] - t[k], obj);   // t_j - t_k from a to b
            if (delta < bestDelta)
            {
                bestDelta = delta; bj = j; bk = k;      // improving move
                if (rule == FIRST) { stop = true; break; }
            }
        }
    }
    if (bj == -1) return false;                         // no improving move: local optimum

    int a = mach[bj], b = mach[bk];                     // apply the move
    L[a] = L[a] - (t[bj] - t[bk]);
    L[b] = L[b] + (t[bj] - t[bk]);
    mach[bj] = b; mach[bk] = a;
    return true;
}

Writing src/PCmaxSwapStep.cpp


## BPP: move and swap

The **move** and **swap steps** for the bin packing problem.

In [23]:
%%writefile src/BPPMoveStep.cpp
#include "LocalSearch.h"

// one step of the move local search for the BPP
// bin[i] is the bin of item i, L[b] the load of bin b (L[b] = 0 if bin b is empty)
// obj = NB (number of bins, minimized) or SQ (sum of the squared loads, maximized)

bool BPPMoveStep(int n, const Vec& w, int c, Vec& bin, Vec& L, int obj, int rule)
{
    int bestGain = 0;                                   // best gain found so far
    int bi = -1, bb = -1;                               // best move found so far
    bool stop = false;
    for (int i = 0; i < n && !stop; ++i)
    {
        int a = bin[i];                                 // bin of item i
        for (int b = 0; b < (int)L.size(); ++b)
        {
            if (b == a || L[b] == 0) continue;          // same bin, or empty bin
            if (L[b] + w[i] > c) continue;              // item i does not fit into bin b
            int gain;                                   // improvement of the objective
            if (obj == SQ)
                gain = 2 * w[i] * (L[b] - L[a] + w[i]);
            else
                gain = (L[a] == w[i]) ? 1 : 0;          // bin a becomes empty
            if (gain > bestGain)
            {
                bestGain = gain; bi = i; bb = b;        // improving move
                if (rule == FIRST) { stop = true; break; }
            }
        }
    }
    if (bi == -1) return false;                         // no improving move: local optimum

    L[bin[bi]] = L[bin[bi]] - w[bi];                    // apply the move
    L[bb] = L[bb] + w[bi];
    bin[bi] = bb;
    return true;
}

Writing src/BPPMoveStep.cpp


In [24]:
%%writefile src/BPPSwapStep.cpp
#include "LocalSearch.h"

// one step of the swap local search for the BPP, with the auxiliary objective
// sum of the squared loads (maximized); a swap never changes the number of bins
// the move (i, k) exchanges the bins of the items i and k

bool BPPSwapStep(int n, const Vec& w, int c, Vec& bin, Vec& L, int rule)
{
    int bestGain = 0;                                   // best gain found so far
    int bi = -1, bk = -1;                               // best move found so far
    bool stop = false;
    for (int i = 0; i < n && !stop; ++i)
    {
        for (int k = i + 1; k < n; ++k)
        {
            int a = bin[i], b = bin[k];
            if (a == b || w[i] == w[k]) continue;       // the loads would not change
            int dw = w[i] - w[k];                       // weight transferred from a to b
            if (L[b] + dw > c || L[a] - dw > c) continue;   // a bin would be overfull
            int gain = 2 * dw * (L[b] - L[a] + dw);     // improvement of the objective
            if (gain > bestGain)
            {
                bestGain = gain; bi = i; bk = k;        // improving move
                if (rule == FIRST) { stop = true; break; }
            }
        }
    }
    if (bi == -1) return false;                         // no improving move: local optimum

    int a = bin[bi], b = bin[bk], dw = w[bi] - w[bk];   // apply the move
    L[a] = L[a] - dw;
    L[b] = L[b] + dw;
    bin[bi] = b; bin[bk] = a;
    return true;
}

Writing src/BPPSwapStep.cpp


## KP: flip+swap and penalty flip

The **flip+swap step** and the **penalty flip step** for the KP (the penalty $\alpha = \alpha_{num}/\alpha_{den}$ is a fraction, and all the values are multiplied by $\alpha_{den}$ to work with integers).

In [25]:
%%writefile src/KPFlipSwapStep.cpp
#include "LocalSearch.h"

// one step of the local search for the KP with the neighbourhood flip + swap,
// restricted to feasible solutions; x[i] = 1 if item i is selected
// the flips are examined first, then the swaps (item i out, item k in)

bool KPFlipSwapStep(int n, const Vec& p, const Vec& w, int c, Vec& x, int rule)
{
    int W = 0;                                          // weight of the current solution
    for (int i = 0; i < n; ++i) W = W + w[i] * x[i];

    int bestDp = 0;                                     // best profit increase found so far
    int bi = -1, bk = -1;                               // best move found so far (bk = -1: flip)
    bool stop = false;
    for (int i = 0; i < n && !stop; ++i)                // flip of item i
    {
        int dp = (1 - 2 * x[i]) * p[i];                 // remove, or add, item i
        int dw = (1 - 2 * x[i]) * w[i];
        if (W + dw <= c && dp > bestDp)
        {
            bestDp = dp; bi = i; bk = -1;               // improving move
            if (rule == FIRST) stop = true;
        }
    }
    for (int i = 0; i < n && !stop; ++i)                // swap: item i out, item k in
    {
        if (x[i] == 0) continue;
        for (int k = 0; k < n; ++k)
        {
            if (x[k] == 1) continue;
            int dp = p[k] - p[i];
            int dw = w[k] - w[i];
            if (W + dw <= c && dp > bestDp)
            {
                bestDp = dp; bi = i; bk = k;            // improving move
                if (rule == FIRST) { stop = true; break; }
            }
        }
    }
    if (bi == -1) return false;                         // no improving move: local optimum

    x[bi] = 1 - x[bi];                                  // apply the flip, or the swap
    if (bk != -1) x[bk] = 1;
    return true;
}

Writing src/KPFlipSwapStep.cpp


In [26]:
%%writefile src/KPPenaltyFlipStep.cpp
#include "LocalSearch.h"

// one step of the flip local search for the KP with a penalty for the excess weight:
// g(x) = profit(x) - alpha * max(0, weight(x) - c), with alpha = alpha_num / alpha_den
// infeasible solutions are allowed; all the values are multiplied by alpha_den

bool KPPenaltyFlipStep(int n, const Vec& p, const Vec& w, int c, int alpha_num, int alpha_den, Vec& x, int rule)
{
    int P = 0, W = 0;                                   // profit and weight of the current solution
    for (int i = 0; i < n; ++i) { P = P + p[i] * x[i]; W = W + w[i] * x[i]; }
    int excess = (W > c) ? W - c : 0;
    int g = alpha_den * P - alpha_num * excess;        // alpha_den * g(x)

    int bestDelta = 0;                                  // best increase of alpha_den * g found so far
    int bi = -1;                                        // best move found so far
    for (int i = 0; i < n; ++i)                         // flip of item i
    {
        int newP = P + (1 - 2 * x[i]) * p[i];
        int newW = W + (1 - 2 * x[i]) * w[i];
        int newExcess = (newW > c) ? newW - c : 0;
        int delta = alpha_den * newP - alpha_num * newExcess - g;
        if (delta > bestDelta)
        {
            bestDelta = delta; bi = i;                  // improving move
            if (rule == FIRST) break;
        }
    }
    if (bi == -1) return false;                         // no improving move: local optimum

    x[bi] = 1 - x[bi];                                  // apply the flip
    return true;
}

Writing src/KPPenaltyFlipStep.cpp


## Multi-start 2-opt

The **multi-start 2-opt** from random tours.

In [27]:
%%writefile src/MultiStartTwoOpt.cpp
#include "LocalSearch.h"

// multi-start local search for the TSP: M random tours, each improved by 2-opt
// the random tours are generated by the Park-Miller generator, starting from seed
// best receives the best local optimum found; the function returns its length

int MultiStartTwoOpt(int n, const Mat& d, int M, long long seed, int rule, Vec& best)
{
    int bestLength = -1;                                // length of the best tour found so far
    for (int s = 1; s <= M; ++s)
    {
        Vec pi = RandomTour(n, seed);                   // random starting tour
        LocalSearch([&]() { return TwoOptStep(n, d, pi, rule); });
        int length = TourLength(n, d, pi);              // length of the local optimum
        if (bestLength == -1 || length < bestLength)
        {
            bestLength = length;                        // new best tour
            best = pi;
        }
    }
    return bestLength;
}

Writing src/MultiStartTwoOpt.cpp


## Execution on the examples of the chapter

Every example of the chapter and every instance of the exercises, with the expected result written in the code: each line shows the result found and `OK` when it is the expected one; the last line counts the checks and the errors.

In [28]:
%%writefile src/Main.cpp
#include <cstdio>
#include <string>
#include <algorithm>
#include <set>
#include "LocalSearch.h"

// runs every example of the lecture notes and every instance of the exercises,
// and compares the values found with those computed by an independent Python engine
// (this file is generated by gen_main.py: do not edit it by hand)

static int errors = 0, checks = 0;

static void check(const char* name, const std::string& found, const std::string& expected)
{
    checks++;
    bool ok = (found == expected);
    if (!ok) errors++;
    printf("%-40s %s\n", name, ok ? "OK" : "<-- DIFFERENT");
    if (!ok) printf("     found    %s\n     expected %s\n", found.c_str(), expected.c_str());
}
static int KPProfit(int n, const Vec& p, const Vec& x)
{ int P = 0; for (int i = 0; i < n; ++i) P += p[i] * x[i]; return P; }
static std::string KPPW(int n, const Vec& p, const Vec& w, const Vec& x)
{ int P = 0, W = 0; for (int i = 0; i < n; ++i) { P += p[i] * x[i]; W += w[i] * x[i]; }
  return std::to_string(P) + "/" + std::to_string(W); }
static std::string BPPVal(int n, const Vec& bin, const Vec& L)
{ int S = 0; for (int l : L) S += l * l; return std::to_string(NumberOfBins(n, bin)) + "/" + std::to_string(S); }
static std::string PCMVal(int m, const Vec& L)
{ int C = 0, S = 0; for (int h = 0; h < m; ++h) { if (L[h] > C) C = L[h]; S += L[h] * L[h]; }
  return std::to_string(C) + "/" + std::to_string(S); }
static std::string Hist(std::vector<int> v)
{ std::sort(v.begin(), v.end()); std::string s; size_t a = 0;
  while (a < v.size()) { size_t b = a; while (b < v.size() && v[b] == v[a]) ++b;
    s += (a ? " " : "") + std::to_string(v[a]) + ":" + std::to_string(b - a); a = b; }
  return s; }
static Vec Canon(Vec pi)
{ int n = (int)pi.size(); Vec c; int k0 = 0; while (pi[k0] != 0) ++k0;
  for (int k = 0; k < n; ++k) c.push_back(pi[(k0 + k) % n]);
  if (c[1] > c[n - 1]) std::reverse(c.begin() + 1, c.end());
  return c; }
static std::string Join(const Vec& v)
{ std::string s; for (size_t k = 0; k < v.size(); ++k) s += (k ? " " : "") + std::to_string(v[k]); return s; }

int main()
{
    { Vec p={10,7,6,4}, w={5,4,3,3}; check("KP ref: GR", Join(GreedyKP(4,p,w,9,{0,2,1,3})), "1 0 1 0");
      check("KP ref: greedy by weight", Join(GreedyKP(4,p,w,9,{2,3,1,0})), "0 0 1 1"); }
    { Vec w={4,4,5,3,2,3}; check("BPP ref: FF", Join(FirstFit(6,w,7,{0,1,2,3,4,5})), "0 1 2 0 1 3");
      check("BPP ref: NF", Join(NextFit(6,w,7)), "0 1 2 3 3 4");
      check("BPP ref: FFD", Join(FirstFit(6,w,7,{2,0,1,3,5,4})), "1 2 0 1 0 2"); }
    { Vec t={5,5,4,4,3,3,3}; check("PCmax ref: LPT", Join(LPT(7,3,t)), "0 1 2 2 0 1 0"); }
    { Mat d={{0,5,2,2,9},{5,0,4,3,4},{2,4,0,4,7},{2,3,4,0,7},{9,4,7,7,0}}; check("TSP ref: NN from city 1", Join(NearestNeighbour(5,d,0)), "0 2 1 3 4"); }
    { Mat d={{0,7,10,12,13,11,7,5,9,4},{7,0,4,7,10,11,7,4,5,3},{10,4,0,4,8,11,7,6,4,6},{12,7,4,0,4,8,6,7,3,9},{13,10,8,4,0,5,6,8,5,11},{11,11,11,8,5,0,4,8,7,11},{7,7,7,6,6,4,0,3,4,6},{5,4,6,7,8,8,3,0,4,3},{9,5,4,3,5,7,4,4,0,6},{4,3,6,9,11,11,6,3,6,0}}; check("TSP10: NN from city 1", Join(NearestNeighbour(10,d,0)), "0 9 1 2 3 8 6 7 4 5");
      std::vector<std::pair<int,int>> pts={{1,1},{3,8},{6,10},{10,9},{13,6},{12,1},{8,3},{5,4},{8,7},{2,5}};
      check("TSP10: distances", std::to_string(EuclideanDistances(pts)==d), "1"); }
    { Vec p={10,7,6,4}, w={5,4,3,3}, x={0,0,1,1}; int n=4, c=9;
      std::string s=std::to_string(KPProfit(n,p,x));
      while (KPFlipSwapStep(n,p,w,c,x,FIRST)) s+=" "+std::to_string(KPProfit(n,p,x));
      check("KP ref from (0,0,1,1) first", s, "10 14 17"); }
    { Vec p={10,7,6,4}, w={5,4,3,3}, x={0,0,1,1}; int n=4, c=9;
      std::string s=std::to_string(KPProfit(n,p,x));
      while (KPFlipSwapStep(n,p,w,c,x,BEST)) s+=" "+std::to_string(KPProfit(n,p,x));
      check("KP ref from (0,0,1,1) best", s, "10 16 17"); }
    { Vec p={10,7,6,4}, w={5,4,3,3}, x={1,0,1,0}; int n=4, c=9;
      std::string s=KPPW(n,p,w,x);
      while (KPPenaltyFlipStep(n,p,w,c,1,1,x,FIRST)) s+=" "+KPPW(n,p,w,x);
      check("KP ref penalty a=1/1 first", s, "16/8 23/12 27/15"); }
    { Vec p={10,7,6,4}, w={5,4,3,3}, x={1,0,1,0}; int n=4, c=9;
      std::string s=KPPW(n,p,w,x);
      while (KPPenaltyFlipStep(n,p,w,c,1,1,x,BEST)) s+=" "+KPPW(n,p,w,x);
      check("KP ref penalty a=1/1 best", s, "16/8 23/12 27/15"); }
    { Vec p={10,7,6,4}, w={5,4,3,3}, x={1,0,1,0}; int n=4, c=9;
      std::string s=KPPW(n,p,w,x);
      while (KPPenaltyFlipStep(n,p,w,c,2,1,x,FIRST)) s+=" "+KPPW(n,p,w,x);
      check("KP ref penalty a=2/1 first", s, "16/8 23/12"); }
    { Vec p={10,7,6,4}, w={5,4,3,3}, x={1,0,1,0}; int n=4, c=9;
      std::string s=KPPW(n,p,w,x);
      while (KPPenaltyFlipStep(n,p,w,c,2,1,x,BEST)) s+=" "+KPPW(n,p,w,x);
      check("KP ref penalty a=2/1 best", s, "16/8 23/12"); }
    { Vec p={10,7,6,4}, w={5,4,3,3}, x={1,0,1,0}; int n=4, c=9;
      std::string s=KPPW(n,p,w,x);
      while (KPPenaltyFlipStep(n,p,w,c,9,4,x,FIRST)) s+=" "+KPPW(n,p,w,x);
      check("KP ref penalty a=9/4 first", s, "16/8 23/12 17/9"); }
    { Vec p={10,7,6,4}, w={5,4,3,3}, x={1,0,1,0}; int n=4, c=9;
      std::string s=KPPW(n,p,w,x);
      while (KPPenaltyFlipStep(n,p,w,c,9,4,x,BEST)) s+=" "+KPPW(n,p,w,x);
      check("KP ref penalty a=9/4 best", s, "16/8 23/12 17/9"); }
    { Vec p={10,7,6,4}, w={5,4,3,3}, x={1,0,1,0}; int n=4, c=9;
      std::string s=KPPW(n,p,w,x);
      while (KPPenaltyFlipStep(n,p,w,c,3,1,x,FIRST)) s+=" "+KPPW(n,p,w,x);
      check("KP ref penalty a=3/1 first", s, "16/8"); }
    { Vec p={10,7,6,4}, w={5,4,3,3}, x={1,0,1,0}; int n=4, c=9;
      std::string s=KPPW(n,p,w,x);
      while (KPPenaltyFlipStep(n,p,w,c,3,1,x,BEST)) s+=" "+KPPW(n,p,w,x);
      check("KP ref penalty a=3/1 best", s, "16/8"); }
    { Vec w={4,4,5,3,2,3}, bin={0,1,2,0,1,3}; int n=6, c=7; Vec L=Loads(n,n,bin,w);
      std::string s=BPPVal(n,bin,L);
      while (BPPMoveStep(n,w,c,bin,L,NB,FIRST)) s+=" "+BPPVal(n,bin,L);
      check("BPP ref FF move nb first", s, "4/119"); }
    { Vec w={4,4,5,3,2,3}, bin={0,1,2,0,1,3}; int n=6, c=7; Vec L=Loads(n,n,bin,w);
      std::string s=BPPVal(n,bin,L);
      while (BPPMoveStep(n,w,c,bin,L,NB,BEST)) s+=" "+BPPVal(n,bin,L);
      check("BPP ref FF move nb best", s, "4/119"); }
    { Vec w={4,4,5,3,2,3}, bin={0,1,2,0,1,3}; int n=6, c=7; Vec L=Loads(n,n,bin,w);
      std::string s=BPPVal(n,bin,L);
      while (BPPMoveStep(n,w,c,bin,L,SQ,FIRST)) s+=" "+BPPVal(n,bin,L);
      check("BPP ref FF move sq first", s, "4/119 4/127 3/147"); }
    { Vec w={4,4,5,3,2,3}, bin={0,1,2,0,1,3}; int n=6, c=7; Vec L=Loads(n,n,bin,w);
      std::string s=BPPVal(n,bin,L);
      while (BPPMoveStep(n,w,c,bin,L,SQ,BEST)) s+=" "+BPPVal(n,bin,L);
      check("BPP ref FF move sq best", s, "4/119 4/127 3/147"); }
    { Vec w={4,4,5,3,2,3}, bin={0,1,2,0,1,3}; int n=6, c=7; Vec L=Loads(n,n,bin,w);
      std::string s=BPPVal(n,bin,L);
      while (BPPSwapStep(n,w,c,bin,L,FIRST)) s+=" "+BPPVal(n,bin,L);
      check("BPP ref FF swap sq first", s, "4/119 4/123"); }
    { Vec w={4,4,5,3,2,3}, bin={0,1,2,0,1,3}; int n=6, c=7; Vec L=Loads(n,n,bin,w);
      std::string s=BPPVal(n,bin,L);
      while (BPPSwapStep(n,w,c,bin,L,BEST)) s+=" "+BPPVal(n,bin,L);
      check("BPP ref FF swap sq best", s, "4/119 4/127"); }
    { Vec w={4,4,5,3,2,3}, bin={0,1,2,0,1,3}; int n=6, c=7; Vec L=Loads(n,n,bin,w);
      std::string s=BPPVal(n,bin,L);
      VND({ [&]() { bool r=BPPMoveStep(n,w,c,bin,L,SQ,FIRST); if (r) s+=" "+BPPVal(n,bin,L); return r; },
            [&]() { bool r=BPPSwapStep(n,w,c,bin,L,FIRST); if (r) s+=" "+BPPVal(n,bin,L); return r; } });
      check("BPP ref FF vnd sq first", s, "4/119 4/127 3/147"); }
    { Vec w={4,4,5,3,2,3}, bin={0,1,2,0,1,3}; int n=6, c=7; Vec L=Loads(n,n,bin,w);
      std::string s=BPPVal(n,bin,L);
      VND({ [&]() { bool r=BPPMoveStep(n,w,c,bin,L,SQ,BEST); if (r) s+=" "+BPPVal(n,bin,L); return r; },
            [&]() { bool r=BPPSwapStep(n,w,c,bin,L,BEST); if (r) s+=" "+BPPVal(n,bin,L); return r; } });
      check("BPP ref FF vnd sq best", s, "4/119 4/127 3/147"); }
    { Vec w={4,4,5,3,2,3}, bin={0,1,2,3,3,4}; int n=6, c=7; Vec L=Loads(n,n,bin,w);
      std::string s=BPPVal(n,bin,L);
      while (BPPMoveStep(n,w,c,bin,L,NB,FIRST)) s+=" "+BPPVal(n,bin,L);
      check("BPP ref NF move nb first", s, "5/91 4/115"); }
    { Vec w={4,4,5,3,2,3}, bin={0,1,2,3,3,4}; int n=6, c=7; Vec L=Loads(n,n,bin,w);
      std::string s=BPPVal(n,bin,L);
      while (BPPMoveStep(n,w,c,bin,L,NB,BEST)) s+=" "+BPPVal(n,bin,L);
      check("BPP ref NF move nb best", s, "5/91 4/115"); }
    { Vec w={4,4,5,3,2,3}, bin={0,1,2,3,3,4}; int n=6, c=7; Vec L=Loads(n,n,bin,w);
      std::string s=BPPVal(n,bin,L);
      while (BPPMoveStep(n,w,c,bin,L,SQ,FIRST)) s+=" "+BPPVal(n,bin,L);
      check("BPP ref NF move sq first", s, "5/91 4/115 4/127 3/147"); }
    { Vec w={4,4,5,3,2,3}, bin={0,1,2,3,3,4}; int n=6, c=7; Vec L=Loads(n,n,bin,w);
      std::string s=BPPVal(n,bin,L);
      while (BPPMoveStep(n,w,c,bin,L,SQ,BEST)) s+=" "+BPPVal(n,bin,L);
      check("BPP ref NF move sq best", s, "5/91 4/115 4/127 3/147"); }
    { Vec w={4,4,5,3,2,3}, bin={0,1,2,3,3,4}; int n=6, c=7; Vec L=Loads(n,n,bin,w);
      std::string s=BPPVal(n,bin,L);
      while (BPPSwapStep(n,w,c,bin,L,FIRST)) s+=" "+BPPVal(n,bin,L);
      check("BPP ref NF swap sq first", s, "5/91 5/95 5/99"); }
    { Vec w={4,4,5,3,2,3}, bin={0,1,2,3,3,4}; int n=6, c=7; Vec L=Loads(n,n,bin,w);
      std::string s=BPPVal(n,bin,L);
      while (BPPSwapStep(n,w,c,bin,L,BEST)) s+=" "+BPPVal(n,bin,L);
      check("BPP ref NF swap sq best", s, "5/91 5/103"); }
    { Vec w={4,4,5,3,2,3}, bin={0,1,2,3,3,4}; int n=6, c=7; Vec L=Loads(n,n,bin,w);
      std::string s=BPPVal(n,bin,L);
      VND({ [&]() { bool r=BPPMoveStep(n,w,c,bin,L,SQ,FIRST); if (r) s+=" "+BPPVal(n,bin,L); return r; },
            [&]() { bool r=BPPSwapStep(n,w,c,bin,L,FIRST); if (r) s+=" "+BPPVal(n,bin,L); return r; } });
      check("BPP ref NF vnd sq first", s, "5/91 4/115 4/127 3/147"); }
    { Vec w={4,4,5,3,2,3}, bin={0,1,2,3,3,4}; int n=6, c=7; Vec L=Loads(n,n,bin,w);
      std::string s=BPPVal(n,bin,L);
      VND({ [&]() { bool r=BPPMoveStep(n,w,c,bin,L,SQ,BEST); if (r) s+=" "+BPPVal(n,bin,L); return r; },
            [&]() { bool r=BPPSwapStep(n,w,c,bin,L,BEST); if (r) s+=" "+BPPVal(n,bin,L); return r; } });
      check("BPP ref NF vnd sq best", s, "5/91 4/115 4/127 3/147"); }
    { Mat d={{0,5,2,2,9},{5,0,4,3,4},{2,4,0,4,7},{2,3,4,0,7},{9,4,7,7,0}}; Vec pi={0,2,1,3,4}; int n=5;
      std::string s=std::to_string(TourLength(n,d,pi));
      while (TwoOptStep(n,d,pi,FIRST)) s+=" "+std::to_string(TourLength(n,d,pi));
      check("TSP ref NN 2-opt first", s, "25 22 18"); }
    { Mat d={{0,5,2,2,9},{5,0,4,3,4},{2,4,0,4,7},{2,3,4,0,7},{9,4,7,7,0}}; Vec pi={0,2,1,3,4}; int n=5;
      std::string s=std::to_string(TourLength(n,d,pi));
      while (TwoOptStep(n,d,pi,BEST)) s+=" "+std::to_string(TourLength(n,d,pi));
      check("TSP ref NN 2-opt best", s, "25 19 18"); }
    { Mat d={{0,7,10,12,13,11,7,5,9,4},{7,0,4,7,10,11,7,4,5,3},{10,4,0,4,8,11,7,6,4,6},{12,7,4,0,4,8,6,7,3,9},{13,10,8,4,0,5,6,8,5,11},{11,11,11,8,5,0,4,8,7,11},{7,7,7,6,6,4,0,3,4,6},{5,4,6,7,8,8,3,0,4,3},{9,5,4,3,5,7,4,4,0,6},{4,3,6,9,11,11,6,3,6,0}}; Vec pi={0,9,1,2,3,8,6,7,4,5}; int n=10;
      std::string s=std::to_string(TourLength(n,d,pi));
      while (TwoOptStep(n,d,pi,FIRST)) s+=" "+std::to_string(TourLength(n,d,pi));
      check("TSP10 NN 2opt first", s, "49 47 46 43 41"); }
    { Mat d={{0,7,10,12,13,11,7,5,9,4},{7,0,4,7,10,11,7,4,5,3},{10,4,0,4,8,11,7,6,4,6},{12,7,4,0,4,8,6,7,3,9},{13,10,8,4,0,5,6,8,5,11},{11,11,11,8,5,0,4,8,7,11},{7,7,7,6,6,4,0,3,4,6},{5,4,6,7,8,8,3,0,4,3},{9,5,4,3,5,7,4,4,0,6},{4,3,6,9,11,11,6,3,6,0}}; Vec pi={0,9,1,2,3,8,6,7,4,5}; int n=10;
      std::string s=std::to_string(TourLength(n,d,pi));
      while (TwoOptStep(n,d,pi,BEST)) s+=" "+std::to_string(TourLength(n,d,pi));
      check("TSP10 NN 2opt best", s, "49 44 40 39"); }
    { Mat d={{0,7,10,12,13,11,7,5,9,4},{7,0,4,7,10,11,7,4,5,3},{10,4,0,4,8,11,7,6,4,6},{12,7,4,0,4,8,6,7,3,9},{13,10,8,4,0,5,6,8,5,11},{11,11,11,8,5,0,4,8,7,11},{7,7,7,6,6,4,0,3,4,6},{5,4,6,7,8,8,3,0,4,3},{9,5,4,3,5,7,4,4,0,6},{4,3,6,9,11,11,6,3,6,0}}; Vec pi={0,9,1,2,3,8,6,7,4,5}; int n=10;
      std::string s=std::to_string(TourLength(n,d,pi));
      while (OrOptStep(n,d,pi,FIRST)) s+=" "+std::to_string(TourLength(n,d,pi));
      check("TSP10 NN or first", s, "49 47 46 40 39"); }
    { Mat d={{0,7,10,12,13,11,7,5,9,4},{7,0,4,7,10,11,7,4,5,3},{10,4,0,4,8,11,7,6,4,6},{12,7,4,0,4,8,6,7,3,9},{13,10,8,4,0,5,6,8,5,11},{11,11,11,8,5,0,4,8,7,11},{7,7,7,6,6,4,0,3,4,6},{5,4,6,7,8,8,3,0,4,3},{9,5,4,3,5,7,4,4,0,6},{4,3,6,9,11,11,6,3,6,0}}; Vec pi={0,9,1,2,3,8,6,7,4,5}; int n=10;
      std::string s=std::to_string(TourLength(n,d,pi));
      while (OrOptStep(n,d,pi,BEST)) s+=" "+std::to_string(TourLength(n,d,pi));
      check("TSP10 NN or best", s, "49 40 39"); }
    { Mat d={{0,7,10,12,13,11,7,5,9,4},{7,0,4,7,10,11,7,4,5,3},{10,4,0,4,8,11,7,6,4,6},{12,7,4,0,4,8,6,7,3,9},{13,10,8,4,0,5,6,8,5,11},{11,11,11,8,5,0,4,8,7,11},{7,7,7,6,6,4,0,3,4,6},{5,4,6,7,8,8,3,0,4,3},{9,5,4,3,5,7,4,4,0,6},{4,3,6,9,11,11,6,3,6,0}}; Vec pi={0,9,1,2,3,8,6,7,4,5}; int n=10;
      std::string s=std::to_string(TourLength(n,d,pi));
      VND({ [&]() { bool r=TwoOptStep(n,d,pi,FIRST); if (r) s+=" "+std::to_string(TourLength(n,d,pi)); return r; },
            [&]() { bool r=OrOptStep(n,d,pi,FIRST); if (r) s+=" "+std::to_string(TourLength(n,d,pi)); return r; } });
      check("TSP10 NN vnd first", s, "49 47 46 43 41 39"); }
    { Mat d={{0,7,10,12,13,11,7,5,9,4},{7,0,4,7,10,11,7,4,5,3},{10,4,0,4,8,11,7,6,4,6},{12,7,4,0,4,8,6,7,3,9},{13,10,8,4,0,5,6,8,5,11},{11,11,11,8,5,0,4,8,7,11},{7,7,7,6,6,4,0,3,4,6},{5,4,6,7,8,8,3,0,4,3},{9,5,4,3,5,7,4,4,0,6},{4,3,6,9,11,11,6,3,6,0}}; Vec pi={0,9,1,2,3,8,6,7,4,5}; int n=10;
      std::string s=std::to_string(TourLength(n,d,pi));
      VND({ [&]() { bool r=TwoOptStep(n,d,pi,BEST); if (r) s+=" "+std::to_string(TourLength(n,d,pi)); return r; },
            [&]() { bool r=OrOptStep(n,d,pi,BEST); if (r) s+=" "+std::to_string(TourLength(n,d,pi)); return r; } });
      check("TSP10 NN vnd best", s, "49 44 40 39"); }
    { Mat d={{0,7,10,12,13,11,7,5,9,4},{7,0,4,7,10,11,7,4,5,3},{10,4,0,4,8,11,7,6,4,6},{12,7,4,0,4,8,6,7,3,9},{13,10,8,4,0,5,6,8,5,11},{11,11,11,8,5,0,4,8,7,11},{7,7,7,6,6,4,0,3,4,6},{5,4,6,7,8,8,3,0,4,3},{9,5,4,3,5,7,4,4,0,6},{4,3,6,9,11,11,6,3,6,0}}; Vec pi={0,9,1,2,3,4,5,6,8,7}; int n=10;
      std::string s=std::to_string(TourLength(n,d,pi));
      while (OrOptStep(n,d,pi,FIRST)) s+=" "+std::to_string(TourLength(n,d,pi));
      check("TSP10 Or-opt from 41 first", s, "41 39"); }
    { Mat d={{0,7,10,12,13,11,7,5,9,4},{7,0,4,7,10,11,7,4,5,3},{10,4,0,4,8,11,7,6,4,6},{12,7,4,0,4,8,6,7,3,9},{13,10,8,4,0,5,6,8,5,11},{11,11,11,8,5,0,4,8,7,11},{7,7,7,6,6,4,0,3,4,6},{5,4,6,7,8,8,3,0,4,3},{9,5,4,3,5,7,4,4,0,6},{4,3,6,9,11,11,6,3,6,0}}; Vec pi={0,9,1,2,3,4,5,6,8,7}; int n=10;
      std::string s=std::to_string(TourLength(n,d,pi));
      while (OrOptStep(n,d,pi,BEST)) s+=" "+std::to_string(TourLength(n,d,pi));
      check("TSP10 Or-opt from 41 best", s, "41 39"); }
    { Vec t={5,5,4,4,3,3,3}, mach={0,1,2,2,0,1,0}; int n=7, m=3; Vec L=Loads(n,m,mach,t);
      std::string s=PCMVal(m,L);
      while (PCmaxMoveStep(n,m,t,mach,L,CMAX,FIRST)) s+=" "+PCMVal(m,L);
      check("PCmax ref LPT move cmax first", s, "11/249"); }
    { Vec t={5,5,4,4,3,3,3}, mach={0,1,2,2,0,1,0}; int n=7, m=3; Vec L=Loads(n,m,mach,t);
      std::string s=PCMVal(m,L);
      while (PCmaxMoveStep(n,m,t,mach,L,CMAX,BEST)) s+=" "+PCMVal(m,L);
      check("PCmax ref LPT move cmax best", s, "11/249"); }
    { Vec t={5,5,4,4,3,3,3}, mach={0,1,2,2,0,1,0}; int n=7, m=3; Vec L=Loads(n,m,mach,t);
      std::string s=PCMVal(m,L);
      while (PCmaxMoveStep(n,m,t,mach,L,SQ,FIRST)) s+=" "+PCMVal(m,L);
      check("PCmax ref LPT move sq first", s, "11/249"); }
    { Vec t={5,5,4,4,3,3,3}, mach={0,1,2,2,0,1,0}; int n=7, m=3; Vec L=Loads(n,m,mach,t);
      std::string s=PCMVal(m,L);
      while (PCmaxMoveStep(n,m,t,mach,L,SQ,BEST)) s+=" "+PCMVal(m,L);
      check("PCmax ref LPT move sq best", s, "11/249"); }
    { Vec t={5,5,4,4,3,3,3}, mach={0,1,2,2,0,1,0}; int n=7, m=3; Vec L=Loads(n,m,mach,t);
      std::string s=PCMVal(m,L);
      while (PCmaxSwapStep(n,m,t,mach,L,CMAX,FIRST)) s+=" "+PCMVal(m,L);
      check("PCmax ref LPT swap cmax first", s, "11/249 10/245 9/243"); }
    { Vec t={5,5,4,4,3,3,3}, mach={0,1,2,2,0,1,0}; int n=7, m=3; Vec L=Loads(n,m,mach,t);
      std::string s=PCMVal(m,L);
      while (PCmaxSwapStep(n,m,t,mach,L,CMAX,BEST)) s+=" "+PCMVal(m,L);
      check("PCmax ref LPT swap cmax best", s, "11/249 10/245 9/243"); }
    { Vec t={5,5,4,4,3,3,3}, mach={0,1,2,2,0,1,0}; int n=7, m=3; Vec L=Loads(n,m,mach,t);
      std::string s=PCMVal(m,L);
      while (PCmaxSwapStep(n,m,t,mach,L,SQ,FIRST)) s+=" "+PCMVal(m,L);
      check("PCmax ref LPT swap sq first", s, "11/249 10/245 9/243"); }
    { Vec t={5,5,4,4,3,3,3}, mach={0,1,2,2,0,1,0}; int n=7, m=3; Vec L=Loads(n,m,mach,t);
      std::string s=PCMVal(m,L);
      while (PCmaxSwapStep(n,m,t,mach,L,SQ,BEST)) s+=" "+PCMVal(m,L);
      check("PCmax ref LPT swap sq best", s, "11/249 10/245 9/243"); }
    { Vec t={5,5,4,4,3,3,3}, mach={0,1,2,2,0,1,0}; int n=7, m=3; Vec L=Loads(n,m,mach,t);
      std::string s=PCMVal(m,L);
      VND({ [&]() { bool r=PCmaxMoveStep(n,m,t,mach,L,CMAX,FIRST); if (r) s+=" "+PCMVal(m,L); return r; },
            [&]() { bool r=PCmaxSwapStep(n,m,t,mach,L,CMAX,FIRST); if (r) s+=" "+PCMVal(m,L); return r; } });
      check("PCmax ref LPT vnd cmax first", s, "11/249 10/245 9/243"); }
    { Vec t={5,5,4,4,3,3,3}, mach={0,1,2,2,0,1,0}; int n=7, m=3; Vec L=Loads(n,m,mach,t);
      std::string s=PCMVal(m,L);
      VND({ [&]() { bool r=PCmaxMoveStep(n,m,t,mach,L,CMAX,BEST); if (r) s+=" "+PCMVal(m,L); return r; },
            [&]() { bool r=PCmaxSwapStep(n,m,t,mach,L,CMAX,BEST); if (r) s+=" "+PCMVal(m,L); return r; } });
      check("PCmax ref LPT vnd cmax best", s, "11/249 10/245 9/243"); }
    { Vec t={5,5,4,4,3,3,3}, mach={0,1,2,2,0,1,0}; int n=7, m=3; Vec L=Loads(n,m,mach,t);
      std::string s=PCMVal(m,L);
      VND({ [&]() { bool r=PCmaxMoveStep(n,m,t,mach,L,SQ,FIRST); if (r) s+=" "+PCMVal(m,L); return r; },
            [&]() { bool r=PCmaxSwapStep(n,m,t,mach,L,SQ,FIRST); if (r) s+=" "+PCMVal(m,L); return r; } });
      check("PCmax ref LPT vnd sq first", s, "11/249 10/245 9/243"); }
    { Vec t={5,5,4,4,3,3,3}, mach={0,1,2,2,0,1,0}; int n=7, m=3; Vec L=Loads(n,m,mach,t);
      std::string s=PCMVal(m,L);
      VND({ [&]() { bool r=PCmaxMoveStep(n,m,t,mach,L,SQ,BEST); if (r) s+=" "+PCMVal(m,L); return r; },
            [&]() { bool r=PCmaxSwapStep(n,m,t,mach,L,SQ,BEST); if (r) s+=" "+PCMVal(m,L); return r; } });
      check("PCmax ref LPT vnd sq best", s, "11/249 10/245 9/243"); }
    { Vec t={11,10,9,8,7,7,6,5,4,4,3,2}, mach={0,1,2,3,3,2,1,0,3,0,1,2}; int n=12, m=4; Vec L=Loads(n,m,mach,t);
      std::string s=PCMVal(m,L);
      while (PCmaxMoveStep(n,m,t,mach,L,CMAX,FIRST)) s+=" "+PCMVal(m,L);
      check("PCM12 LPT move cmax first", s, "20/1446"); }
    { Vec t={11,10,9,8,7,7,6,5,4,4,3,2}, mach={0,1,2,3,3,2,1,0,3,0,1,2}; int n=12, m=4; Vec L=Loads(n,m,mach,t);
      std::string s=PCMVal(m,L);
      while (PCmaxMoveStep(n,m,t,mach,L,CMAX,BEST)) s+=" "+PCMVal(m,L);
      check("PCM12 LPT move cmax best", s, "20/1446"); }
    { Vec t={11,10,9,8,7,7,6,5,4,4,3,2}, mach={0,1,2,3,3,2,1,0,3,0,1,2}; int n=12, m=4; Vec L=Loads(n,m,mach,t);
      std::string s=PCMVal(m,L);
      while (PCmaxSwapStep(n,m,t,mach,L,CMAX,FIRST)) s+=" "+PCMVal(m,L);
      check("PCM12 LPT swap cmax first", s, "20/1446"); }
    { Vec t={11,10,9,8,7,7,6,5,4,4,3,2}, mach={0,1,2,3,3,2,1,0,3,0,1,2}; int n=12, m=4; Vec L=Loads(n,m,mach,t);
      std::string s=PCMVal(m,L);
      while (PCmaxSwapStep(n,m,t,mach,L,CMAX,BEST)) s+=" "+PCMVal(m,L);
      check("PCM12 LPT swap cmax best", s, "20/1446"); }
    { Vec t={11,10,9,8,7,7,6,5,4,4,3,2}; int n=12, m=4; long long seed=1; std::vector<int> res;
      for (int s=0;s<100;++s) { Vec mach=RandomAssignment(n,m,seed); Vec L=Loads(n,m,mach,t);
        LocalSearch([&]() { return PCmaxMoveStep(n,m,t,mach,L,CMAX,BEST); });
        int C=0; for (int h=0;h<m;++h) if (L[h]>C) C=L[h]; res.push_back(C); }
      check("PCM12 100 starts move cmax", Hist(res), "19:3 20:48 21:29 22:12 23:3 25:2 26:2 33:1"); }
    { Vec t={11,10,9,8,7,7,6,5,4,4,3,2}; int n=12, m=4; long long seed=1; std::vector<int> res;
      for (int s=0;s<100;++s) { Vec mach=RandomAssignment(n,m,seed); Vec L=Loads(n,m,mach,t);
        LocalSearch([&]() { return PCmaxMoveStep(n,m,t,mach,L,SQ,BEST); });
        int C=0; for (int h=0;h<m;++h) if (L[h]>C) C=L[h]; res.push_back(C); }
      check("PCM12 100 starts move sq", Hist(res), "19:4 20:61 21:27 22:8"); }
    { Vec t={11,10,9,8,7,7,6,5,4,4,3,2}; int n=12, m=4; long long seed=1; std::vector<int> res;
      for (int s=0;s<100;++s) { Vec mach=RandomAssignment(n,m,seed); Vec L=Loads(n,m,mach,t);
        LocalSearch([&]() { return PCmaxSwapStep(n,m,t,mach,L,CMAX,BEST); });
        int C=0; for (int h=0;h<m;++h) if (L[h]>C) C=L[h]; res.push_back(C); }
      check("PCM12 100 starts swap cmax", Hist(res), "19:7 20:10 21:6 22:15 23:11 24:12 25:10 26:5 27:3 28:8 29:2 30:2 31:5 33:1 34:1 38:2"); }
    { Vec t={11,10,9,8,7,7,6,5,4,4,3,2}; int n=12, m=4; long long seed=1; std::vector<int> res;
      for (int s=0;s<100;++s) { Vec mach=RandomAssignment(n,m,seed); Vec L=Loads(n,m,mach,t);
        LocalSearch([&]() { return PCmaxSwapStep(n,m,t,mach,L,SQ,BEST); });
        int C=0; for (int h=0;h<m;++h) if (L[h]>C) C=L[h]; res.push_back(C); }
      check("PCM12 100 starts swap sq", Hist(res), "19:20 20:15 22:17 23:14 24:11 26:6 28:10 31:4 33:1 38:2"); }
    { Vec t={11,10,9,8,7,7,6,5,4,4,3,2}; int n=12, m=4; long long seed=1; std::vector<int> res;
      for (int s=0;s<100;++s) { Vec mach=RandomAssignment(n,m,seed); Vec L=Loads(n,m,mach,t);
        VND({ [&]() { return PCmaxMoveStep(n,m,t,mach,L,CMAX,BEST); }, [&]() { return PCmaxSwapStep(n,m,t,mach,L,CMAX,BEST); } });
        int C=0; for (int h=0;h<m;++h) if (L[h]>C) C=L[h]; res.push_back(C); }
      check("PCM12 100 starts vnd cmax", Hist(res), "19:15 20:52 21:17 22:8 23:3 25:2 26:2 33:1"); }
    { Vec t={11,10,9,8,7,7,6,5,4,4,3,2}; int n=12, m=4; long long seed=1; std::vector<int> res;
      for (int s=0;s<100;++s) { Vec mach=RandomAssignment(n,m,seed); Vec L=Loads(n,m,mach,t);
        VND({ [&]() { return PCmaxMoveStep(n,m,t,mach,L,SQ,BEST); }, [&]() { return PCmaxSwapStep(n,m,t,mach,L,SQ,BEST); } });
        int C=0; for (int h=0;h<m;++h) if (L[h]>C) C=L[h]; res.push_back(C); }
      check("PCM12 100 starts vnd sq", Hist(res), "19:68 20:32"); }
    { Vec w={7,6,6,5,5,4,4,4,3,3,2,1}; int n=12, c=10; long long seed=1; std::vector<int> res;
      for (int s=0;s<100;++s) { Vec bin=FirstFit(n,w,c,RandomPermutation(n,seed)); Vec L=Loads(n,n,bin,w);
        
        res.push_back(NumberOfBins(n,bin)); }
      check("BPP12 100 starts start nb", Hist(res), "5:14 6:85 7:1"); }
    { Vec w={7,6,6,5,5,4,4,4,3,3,2,1}; int n=12, c=10; long long seed=1; std::vector<int> res;
      for (int s=0;s<100;++s) { Vec bin=FirstFit(n,w,c,RandomPermutation(n,seed)); Vec L=Loads(n,n,bin,w);
        LocalSearch([&]() { return BPPMoveStep(n,w,c,bin,L,NB,BEST); });
        res.push_back(NumberOfBins(n,bin)); }
      check("BPP12 100 starts move nb", Hist(res), "5:14 6:85 7:1"); }
    { Vec w={7,6,6,5,5,4,4,4,3,3,2,1}; int n=12, c=10; long long seed=1; std::vector<int> res;
      for (int s=0;s<100;++s) { Vec bin=FirstFit(n,w,c,RandomPermutation(n,seed)); Vec L=Loads(n,n,bin,w);
        LocalSearch([&]() { return BPPMoveStep(n,w,c,bin,L,SQ,BEST); });
        res.push_back(NumberOfBins(n,bin)); }
      check("BPP12 100 starts move sq", Hist(res), "5:76 6:24"); }
    { Vec w={7,6,6,5,5,4,4,4,3,3,2,1}; int n=12, c=10; long long seed=1; std::vector<int> res;
      for (int s=0;s<100;++s) { Vec bin=FirstFit(n,w,c,RandomPermutation(n,seed)); Vec L=Loads(n,n,bin,w);
        LocalSearch([&]() { return BPPSwapStep(n,w,c,bin,L,BEST); });
        res.push_back(NumberOfBins(n,bin)); }
      check("BPP12 100 starts swap sq", Hist(res), "5:14 6:85 7:1"); }
    { Vec w={7,6,6,5,5,4,4,4,3,3,2,1}; int n=12, c=10; long long seed=1; std::vector<int> res;
      for (int s=0;s<100;++s) { Vec bin=FirstFit(n,w,c,RandomPermutation(n,seed)); Vec L=Loads(n,n,bin,w);
        VND({ [&]() { return BPPMoveStep(n,w,c,bin,L,SQ,BEST); }, [&]() { return BPPSwapStep(n,w,c,bin,L,BEST); } });
        res.push_back(NumberOfBins(n,bin)); }
      check("BPP12 100 starts vnd sq", Hist(res), "5:89 6:11"); }
    { Mat d={{0,7,10,12,13,11,7,5,9,4},{7,0,4,7,10,11,7,4,5,3},{10,4,0,4,8,11,7,6,4,6},{12,7,4,0,4,8,6,7,3,9},{13,10,8,4,0,5,6,8,5,11},{11,11,11,8,5,0,4,8,7,11},{7,7,7,6,6,4,0,3,4,6},{5,4,6,7,8,8,3,0,4,3},{9,5,4,3,5,7,4,4,0,6},{4,3,6,9,11,11,6,3,6,0}}; int n=(int)d.size(); long long seed=1; std::vector<int> res;
      for (int s=0;s<100;++s) { Vec pi=RandomTour(n,seed); LocalSearch([&]() { return TwoOptStep(n,d,pi,FIRST); }); res.push_back(TourLength(n,d,pi)); }
      check("TSP10 100 starts 2-opt first", Hist(res), "39:46 41:54");
      Vec best; check("TSP10 MultiStartTwoOpt first", std::to_string(MultiStartTwoOpt(n,d,100,1,FIRST,best)), "39"); }
    { Mat d={{0,7,10,12,13,11,7,5,9,4},{7,0,4,7,10,11,7,4,5,3},{10,4,0,4,8,11,7,6,4,6},{12,7,4,0,4,8,6,7,3,9},{13,10,8,4,0,5,6,8,5,11},{11,11,11,8,5,0,4,8,7,11},{7,7,7,6,6,4,0,3,4,6},{5,4,6,7,8,8,3,0,4,3},{9,5,4,3,5,7,4,4,0,6},{4,3,6,9,11,11,6,3,6,0}}; int n=(int)d.size(); long long seed=1; std::vector<int> res;
      for (int s=0;s<100;++s) { Vec pi=RandomTour(n,seed); LocalSearch([&]() { return TwoOptStep(n,d,pi,BEST); }); res.push_back(TourLength(n,d,pi)); }
      check("TSP10 100 starts 2-opt best", Hist(res), "39:87 41:8 43:5");
      Vec best; check("TSP10 MultiStartTwoOpt best", std::to_string(MultiStartTwoOpt(n,d,100,1,BEST,best)), "39"); }
    { std::vector<std::pair<int,int>> pts; long long s0=12345; for (int k=0;k<30;++k) { int a=RandomInt(s0,101); int b=RandomInt(s0,101); pts.push_back({a,b}); } Mat d=EuclideanDistances(pts); int n=(int)d.size(); long long seed=1; std::vector<int> res;
      for (int s=0;s<100;++s) { Vec pi=RandomTour(n,seed); LocalSearch([&]() { return TwoOptStep(n,d,pi,FIRST); }); res.push_back(TourLength(n,d,pi)); }
      check("TSP30 100 starts 2-opt first", Hist(res), "478:14 479:26 480:9 481:6 482:2 484:2 485:1 486:1 488:2 489:1 490:1 493:2 494:2 495:4 496:4 497:5 501:1 502:3 504:1 506:1 508:2 510:1 514:2 515:2 517:1 518:1 519:1 520:1 537:1");
      Vec best; check("TSP30 MultiStartTwoOpt first", std::to_string(MultiStartTwoOpt(n,d,100,1,FIRST,best)), "478"); }
    { std::vector<std::pair<int,int>> pts; long long s0=12345; for (int k=0;k<30;++k) { int a=RandomInt(s0,101); int b=RandomInt(s0,101); pts.push_back({a,b}); } Mat d=EuclideanDistances(pts); int n=(int)d.size(); long long seed=1; std::vector<int> res;
      for (int s=0;s<100;++s) { Vec pi=RandomTour(n,seed); LocalSearch([&]() { return TwoOptStep(n,d,pi,BEST); }); res.push_back(TourLength(n,d,pi)); }
      check("TSP30 100 starts 2-opt best", Hist(res), "478:12 479:34 480:5 481:8 482:1 484:2 485:1 486:1 487:3 489:1 490:1 491:2 492:1 493:2 494:1 495:6 497:1 500:1 501:4 502:2 503:3 508:2 512:2 513:1 515:1 520:1 567:1");
      Vec best; check("TSP30 MultiStartTwoOpt best", std::to_string(MultiStartTwoOpt(n,d,100,1,BEST,best)), "478"); }
    { Vec p={10,7,13,14,8}, w={2,3,8,9,7}, x={1,1,1,0,0}; int n=5, c=18;
      std::string s=std::to_string(KPProfit(n,p,x));
      while (KPFlipSwapStep(n,p,w,c,x,FIRST)) s+=" "+std::to_string(KPProfit(n,p,x));
      check("EX_KP_1 from GR first", s, "30 31 32"); }
    { Vec p={10,7,13,14,8}, w={2,3,8,9,7}, x={1,1,1,0,0}; int n=5, c=18;
      std::string s=std::to_string(KPProfit(n,p,x));
      while (KPFlipSwapStep(n,p,w,c,x,BEST)) s+=" "+std::to_string(KPProfit(n,p,x));
      check("EX_KP_1 from GR best", s, "30 31 32"); }
    { Vec p={10,7,13,14,8}, w={2,3,8,9,7}, x={0,0,1,1,0}; int n=5, c=18;
      std::string s=std::to_string(KPProfit(n,p,x));
      while (KPFlipSwapStep(n,p,w,c,x,BEST)) s+=" "+std::to_string(KPProfit(n,p,x));
      check("EX_KP_1 from greedy by profit best", s, "27"); }
    { Vec p={8,9,14,11,9}, w={3,5,9,9,8}, x={1,1,0,0,0}; int n=5, c=13;
      std::string s=std::to_string(KPProfit(n,p,x));
      while (KPFlipSwapStep(n,p,w,c,x,FIRST)) s+=" "+std::to_string(KPProfit(n,p,x));
      check("EX_KP_2 from GR first", s, "17 18"); }
    { Vec p={8,9,14,11,9}, w={3,5,9,9,8}, x={1,1,0,0,0}; int n=5, c=13;
      std::string s=std::to_string(KPProfit(n,p,x));
      while (KPFlipSwapStep(n,p,w,c,x,BEST)) s+=" "+std::to_string(KPProfit(n,p,x));
      check("EX_KP_2 from GR best", s, "17 22"); }
    { Vec p={14,7,10,8,6}, w={4,4,7,7,9}, x={1,1,0,0,0}; int n=5, c=11;
      std::string s=KPPW(n,p,w,x);
      while (KPPenaltyFlipStep(n,p,w,c,1,1,x,BEST)) s+=" "+KPPW(n,p,w,x);
      check("EX_KP_3 penalty a=1 best", s, "21/8 31/15 39/22"); }
    { Vec p={14,7,10,8,6}, w={4,4,7,7,9}, x={1,1,0,0,0}; int n=5, c=11;
      std::string s=KPPW(n,p,w,x);
      while (KPPenaltyFlipStep(n,p,w,c,2,1,x,BEST)) s+=" "+KPPW(n,p,w,x);
      check("EX_KP_3 penalty a=2 best", s, "21/8 31/15 24/11"); }
    { Vec p={14,7,10,8,6}, w={4,4,7,7,9}, x={1,1,0,0,0}; int n=5, c=11;
      std::string s=KPPW(n,p,w,x);
      while (KPPenaltyFlipStep(n,p,w,c,3,1,x,BEST)) s+=" "+KPPW(n,p,w,x);
      check("EX_KP_3 penalty a=3 best", s, "21/8"); }
    { Vec p={14,7,10,8,6}, w={4,4,7,7,9}, x={1,1,0,0,0}; int n=5, c=11;
      std::string s=KPPW(n,p,w,x);
      while (KPPenaltyFlipStep(n,p,w,c,15,1,x,BEST)) s+=" "+KPPW(n,p,w,x);
      check("EX_KP_3 penalty a=15 best", s, "21/8"); }
    { Vec w={2,3,2,7,4,7,4}, bin={0,0,0,1,2,3,2}; int n=7, c=10; Vec L=Loads(n,n,bin,w);
      std::string s=BPPVal(n,bin,L);
      while (BPPMoveStep(n,w,c,bin,L,NB,FIRST)) s+=" "+BPPVal(n,bin,L);
      check("EX_BPP_1 FF move nb first", s, "4/211"); }
    { Vec w={2,3,2,7,4,7,4}, bin={0,0,0,1,2,3,2}; int n=7, c=10; Vec L=Loads(n,n,bin,w);
      std::string s=BPPVal(n,bin,L);
      while (BPPMoveStep(n,w,c,bin,L,NB,BEST)) s+=" "+BPPVal(n,bin,L);
      check("EX_BPP_1 FF move nb best", s, "4/211"); }
    { Vec w={2,3,2,7,4,7,4}, bin={0,0,0,1,2,3,2}; int n=7, c=10; Vec L=Loads(n,n,bin,w);
      std::string s=BPPVal(n,bin,L);
      while (BPPMoveStep(n,w,c,bin,L,SQ,FIRST)) s+=" "+BPPVal(n,bin,L);
      check("EX_BPP_1 FF move sq first", s, "4/211 4/219 4/223 4/253 3/281"); }
    { Vec w={2,3,2,7,4,7,4}, bin={0,0,0,1,2,3,2}; int n=7, c=10; Vec L=Loads(n,n,bin,w);
      std::string s=BPPVal(n,bin,L);
      while (BPPMoveStep(n,w,c,bin,L,SQ,BEST)) s+=" "+BPPVal(n,bin,L);
      check("EX_BPP_1 FF move sq best", s, "4/211 4/229 4/253 3/281"); }
    { Vec w={2,3,4,5,4,2}, bin={0,0,0,1,1,2}; int n=6, c=10; Vec L=Loads(n,n,bin,w);
      std::string s=BPPVal(n,bin,L);
      while (BPPMoveStep(n,w,c,bin,L,SQ,BEST)) s+=" "+BPPVal(n,bin,L);
      check("EX_BPP_2 FF move sq best", s, "3/166"); }
    { Vec w={2,3,4,5,4,2}, bin={0,0,0,1,1,2}; int n=6, c=10; Vec L=Loads(n,n,bin,w);
      std::string s=BPPVal(n,bin,L);
      while (BPPSwapStep(n,w,c,bin,L,BEST)) s+=" "+BPPVal(n,bin,L);
      check("EX_BPP_2 FF swap sq best", s, "3/166 3/168"); }
    { Vec w={2,3,4,5,4,2}, bin={0,0,0,1,1,2}; int n=6, c=10; Vec L=Loads(n,n,bin,w);
      std::string s=BPPVal(n,bin,L);
      VND({ [&]() { bool r=BPPMoveStep(n,w,c,bin,L,SQ,BEST); if (r) s+=" "+BPPVal(n,bin,L); return r; },
            [&]() { bool r=BPPSwapStep(n,w,c,bin,L,BEST); if (r) s+=" "+BPPVal(n,bin,L); return r; } });
      check("EX_BPP_2 FF vnd sq best", s, "3/166 3/168 2/200"); }
    { Vec w={5,3,3,2,8,4,4}, bin={0,0,1,0,2,1,3}; int n=7, c=10; Vec L=Loads(n,n,bin,w);
      std::string s=BPPVal(n,bin,L);
      while (BPPMoveStep(n,w,c,bin,L,SQ,FIRST)) s+=" "+BPPVal(n,bin,L);
      check("EX_BPP_3 FF move sq first", s, "4/229 4/237"); }
    { Vec w={5,3,3,2,8,4,4}, bin={0,0,1,0,2,1,3}; int n=7, c=10; Vec L=Loads(n,n,bin,w);
      std::string s=BPPVal(n,bin,L);
      while (BPPMoveStep(n,w,c,bin,L,SQ,BEST)) s+=" "+BPPVal(n,bin,L);
      check("EX_BPP_3 FF move sq best", s, "4/229 4/237"); }
    { Vec w={5,3,3,2,8,4,4}, bin={0,0,1,0,2,1,3}; int n=7, c=10; Vec L=Loads(n,n,bin,w);
      std::string s=BPPVal(n,bin,L);
      VND({ [&]() { bool r=BPPMoveStep(n,w,c,bin,L,SQ,FIRST); if (r) s+=" "+BPPVal(n,bin,L); return r; },
            [&]() { bool r=BPPSwapStep(n,w,c,bin,L,FIRST); if (r) s+=" "+BPPVal(n,bin,L); return r; } });
      check("EX_BPP_3 FF vnd sq first", s, "4/229 4/237"); }
    { Vec w={5,3,3,2,8,4,4}, bin={0,0,1,0,2,1,3}; int n=7, c=10; Vec L=Loads(n,n,bin,w);
      std::string s=BPPVal(n,bin,L);
      VND({ [&]() { bool r=BPPMoveStep(n,w,c,bin,L,SQ,BEST); if (r) s+=" "+BPPVal(n,bin,L); return r; },
            [&]() { bool r=BPPSwapStep(n,w,c,bin,L,BEST); if (r) s+=" "+BPPVal(n,bin,L); return r; } });
      check("EX_BPP_3 FF vnd sq best", s, "4/229 4/237"); }
    { Mat d={{0,7,6,3,4,6},{7,0,6,5,7,4},{6,6,0,5,5,2},{3,5,5,0,5,3},{4,7,5,5,0,4},{6,4,2,3,4,0}}; Vec pi={0,3,5,2,4,1}; int n=6;
      std::string s=std::to_string(TourLength(n,d,pi));
      while (TwoOptStep(n,d,pi,FIRST)) s+=" "+std::to_string(TourLength(n,d,pi));
      check("EX_TSP_1 NN 2opt first", s, "27 26 23"); }
    { Mat d={{0,7,6,3,4,6},{7,0,6,5,7,4},{6,6,0,5,5,2},{3,5,5,0,5,3},{4,7,5,5,0,4},{6,4,2,3,4,0}}; Vec pi={0,3,5,2,4,1}; int n=6;
      std::string s=std::to_string(TourLength(n,d,pi));
      while (TwoOptStep(n,d,pi,BEST)) s+=" "+std::to_string(TourLength(n,d,pi));
      check("EX_TSP_1 NN 2opt best", s, "27 25 24 23"); }
    { Mat d={{0,7,6,3,4,6},{7,0,6,5,7,4},{6,6,0,5,5,2},{3,5,5,0,5,3},{4,7,5,5,0,4},{6,4,2,3,4,0}}; Vec pi={0,3,5,2,4,1}; int n=6;
      std::string s=std::to_string(TourLength(n,d,pi));
      while (OrOptStep(n,d,pi,FIRST)) s+=" "+std::to_string(TourLength(n,d,pi));
      check("EX_TSP_1 NN or first", s, "27 25 23"); }
    { Mat d={{0,7,6,3,4,6},{7,0,6,5,7,4},{6,6,0,5,5,2},{3,5,5,0,5,3},{4,7,5,5,0,4},{6,4,2,3,4,0}}; Vec pi={0,3,5,2,4,1}; int n=6;
      std::string s=std::to_string(TourLength(n,d,pi));
      while (OrOptStep(n,d,pi,BEST)) s+=" "+std::to_string(TourLength(n,d,pi));
      check("EX_TSP_1 NN or best", s, "27 23"); }
    { Mat d={{0,7,6,3,4,6},{7,0,6,5,7,4},{6,6,0,5,5,2},{3,5,5,0,5,3},{4,7,5,5,0,4},{6,4,2,3,4,0}}; Vec pi={0,3,5,2,4,1}; int n=6;
      std::string s=std::to_string(TourLength(n,d,pi));
      VND({ [&]() { bool r=TwoOptStep(n,d,pi,FIRST); if (r) s+=" "+std::to_string(TourLength(n,d,pi)); return r; },
            [&]() { bool r=OrOptStep(n,d,pi,FIRST); if (r) s+=" "+std::to_string(TourLength(n,d,pi)); return r; } });
      check("EX_TSP_1 NN vnd first", s, "27 26 23"); }
    { Mat d={{0,7,6,3,4,6},{7,0,6,5,7,4},{6,6,0,5,5,2},{3,5,5,0,5,3},{4,7,5,5,0,4},{6,4,2,3,4,0}}; Vec pi={0,3,5,2,4,1}; int n=6;
      std::string s=std::to_string(TourLength(n,d,pi));
      VND({ [&]() { bool r=TwoOptStep(n,d,pi,BEST); if (r) s+=" "+std::to_string(TourLength(n,d,pi)); return r; },
            [&]() { bool r=OrOptStep(n,d,pi,BEST); if (r) s+=" "+std::to_string(TourLength(n,d,pi)); return r; } });
      check("EX_TSP_1 NN vnd best", s, "27 25 24 23"); }
    { Mat d={{0,8,5,6,5,6},{8,0,6,7,7,6},{5,6,0,4,8,2},{6,7,4,0,6,5},{5,7,8,6,0,7},{6,6,2,5,7,0}}; Vec pi={0,2,5,3,4,1}; int n=6;
      std::string s=std::to_string(TourLength(n,d,pi));
      while (TwoOptStep(n,d,pi,FIRST)) s+=" "+std::to_string(TourLength(n,d,pi));
      check("EX_TSP_2 NN 2opt first", s, "33 32 31 30"); }
    { Mat d={{0,8,5,6,5,6},{8,0,6,7,7,6},{5,6,0,4,8,2},{6,7,4,0,6,5},{5,7,8,6,0,7},{6,6,2,5,7,0}}; Vec pi={0,2,5,3,4,1}; int n=6;
      std::string s=std::to_string(TourLength(n,d,pi));
      while (TwoOptStep(n,d,pi,BEST)) s+=" "+std::to_string(TourLength(n,d,pi));
      check("EX_TSP_2 NN 2opt best", s, "33 31"); }
    { Mat d={{0,8,5,6,5,6},{8,0,6,7,7,6},{5,6,0,4,8,2},{6,7,4,0,6,5},{5,7,8,6,0,7},{6,6,2,5,7,0}}; Vec pi={0,2,5,3,4,1}; int n=6;
      std::string s=std::to_string(TourLength(n,d,pi));
      while (OrOptStep(n,d,pi,FIRST)) s+=" "+std::to_string(TourLength(n,d,pi));
      check("EX_TSP_2 NN or first", s, "33 31 30"); }
    { Mat d={{0,8,5,6,5,6},{8,0,6,7,7,6},{5,6,0,4,8,2},{6,7,4,0,6,5},{5,7,8,6,0,7},{6,6,2,5,7,0}}; Vec pi={0,2,5,3,4,1}; int n=6;
      std::string s=std::to_string(TourLength(n,d,pi));
      while (OrOptStep(n,d,pi,BEST)) s+=" "+std::to_string(TourLength(n,d,pi));
      check("EX_TSP_2 NN or best", s, "33 31 30"); }
    { Mat d={{0,8,5,6,5,6},{8,0,6,7,7,6},{5,6,0,4,8,2},{6,7,4,0,6,5},{5,7,8,6,0,7},{6,6,2,5,7,0}}; Vec pi={0,2,5,3,4,1}; int n=6;
      std::string s=std::to_string(TourLength(n,d,pi));
      VND({ [&]() { bool r=TwoOptStep(n,d,pi,FIRST); if (r) s+=" "+std::to_string(TourLength(n,d,pi)); return r; },
            [&]() { bool r=OrOptStep(n,d,pi,FIRST); if (r) s+=" "+std::to_string(TourLength(n,d,pi)); return r; } });
      check("EX_TSP_2 NN vnd first", s, "33 32 31 30"); }
    { Mat d={{0,8,5,6,5,6},{8,0,6,7,7,6},{5,6,0,4,8,2},{6,7,4,0,6,5},{5,7,8,6,0,7},{6,6,2,5,7,0}}; Vec pi={0,2,5,3,4,1}; int n=6;
      std::string s=std::to_string(TourLength(n,d,pi));
      VND({ [&]() { bool r=TwoOptStep(n,d,pi,BEST); if (r) s+=" "+std::to_string(TourLength(n,d,pi)); return r; },
            [&]() { bool r=OrOptStep(n,d,pi,BEST); if (r) s+=" "+std::to_string(TourLength(n,d,pi)); return r; } });
      check("EX_TSP_2 NN vnd best", s, "33 31 30"); }
    { Mat d={{0,7,8,8,9,9},{7,0,7,4,9,8},{8,7,0,7,3,9},{8,4,7,0,5,6},{9,9,3,5,0,7},{9,8,9,6,7,0}}; Vec pi={0,1,3,4,2,5}; int n=6;
      std::string s=std::to_string(TourLength(n,d,pi));
      while (TwoOptStep(n,d,pi,FIRST)) s+=" "+std::to_string(TourLength(n,d,pi));
      check("EX_TSP_3 NN 2opt first", s, "37"); }
    { Mat d={{0,7,8,8,9,9},{7,0,7,4,9,8},{8,7,0,7,3,9},{8,4,7,0,5,6},{9,9,3,5,0,7},{9,8,9,6,7,0}}; Vec pi={0,1,3,4,2,5}; int n=6;
      std::string s=std::to_string(TourLength(n,d,pi));
      while (TwoOptStep(n,d,pi,BEST)) s+=" "+std::to_string(TourLength(n,d,pi));
      check("EX_TSP_3 NN 2opt best", s, "37"); }
    { Mat d={{0,7,8,8,9,9},{7,0,7,4,9,8},{8,7,0,7,3,9},{8,4,7,0,5,6},{9,9,3,5,0,7},{9,8,9,6,7,0}}; Vec pi={0,1,3,4,2,5}; int n=6;
      std::string s=std::to_string(TourLength(n,d,pi));
      while (OrOptStep(n,d,pi,FIRST)) s+=" "+std::to_string(TourLength(n,d,pi));
      check("EX_TSP_3 NN or first", s, "37 35"); }
    { Mat d={{0,7,8,8,9,9},{7,0,7,4,9,8},{8,7,0,7,3,9},{8,4,7,0,5,6},{9,9,3,5,0,7},{9,8,9,6,7,0}}; Vec pi={0,1,3,4,2,5}; int n=6;
      std::string s=std::to_string(TourLength(n,d,pi));
      while (OrOptStep(n,d,pi,BEST)) s+=" "+std::to_string(TourLength(n,d,pi));
      check("EX_TSP_3 NN or best", s, "37 35"); }
    { Mat d={{0,7,8,8,9,9},{7,0,7,4,9,8},{8,7,0,7,3,9},{8,4,7,0,5,6},{9,9,3,5,0,7},{9,8,9,6,7,0}}; Vec pi={0,1,3,4,2,5}; int n=6;
      std::string s=std::to_string(TourLength(n,d,pi));
      VND({ [&]() { bool r=TwoOptStep(n,d,pi,FIRST); if (r) s+=" "+std::to_string(TourLength(n,d,pi)); return r; },
            [&]() { bool r=OrOptStep(n,d,pi,FIRST); if (r) s+=" "+std::to_string(TourLength(n,d,pi)); return r; } });
      check("EX_TSP_3 NN vnd first", s, "37 35"); }
    { Mat d={{0,7,8,8,9,9},{7,0,7,4,9,8},{8,7,0,7,3,9},{8,4,7,0,5,6},{9,9,3,5,0,7},{9,8,9,6,7,0}}; Vec pi={0,1,3,4,2,5}; int n=6;
      std::string s=std::to_string(TourLength(n,d,pi));
      VND({ [&]() { bool r=TwoOptStep(n,d,pi,BEST); if (r) s+=" "+std::to_string(TourLength(n,d,pi)); return r; },
            [&]() { bool r=OrOptStep(n,d,pi,BEST); if (r) s+=" "+std::to_string(TourLength(n,d,pi)); return r; } });
      check("EX_TSP_3 NN vnd best", s, "37 35"); }
    { Vec t={5,7,3,4,4,9}, mach={0,1,2,2,0,1}; int n=6, m=3; Vec L=Loads(n,m,mach,t);
      std::string s=PCMVal(m,L);
      while (PCmaxMoveStep(n,m,t,mach,L,CMAX,FIRST)) s+=" "+PCMVal(m,L);
      check("EX_PCM_1 LIST move cmax first", s, "16/386 14/358 12/346"); }
    { Vec t={5,7,3,4,4,9}, mach={0,1,2,2,0,1}; int n=6, m=3; Vec L=Loads(n,m,mach,t);
      std::string s=PCMVal(m,L);
      while (PCmaxMoveStep(n,m,t,mach,L,CMAX,BEST)) s+=" "+PCMVal(m,L);
      check("EX_PCM_1 LIST move cmax best", s, "16/386 14/358 12/346"); }
    { Vec t={8,4,6,7,5,9,8}, mach={1,0,2,1,0,0,2}; int n=7, m=3; Vec L=Loads(n,m,mach,t);
      std::string s=PCMVal(m,L);
      while (PCmaxMoveStep(n,m,t,mach,L,CMAX,BEST)) s+=" "+PCMVal(m,L);
      check("EX_PCM_2 LPT move cmax best", s, "18/745"); }
    { Vec t={8,4,6,7,5,9,8}, mach={1,0,2,1,0,0,2}; int n=7, m=3; Vec L=Loads(n,m,mach,t);
      std::string s=PCMVal(m,L);
      while (PCmaxSwapStep(n,m,t,mach,L,CMAX,BEST)) s+=" "+PCMVal(m,L);
      check("EX_PCM_2 LPT swap cmax best", s, "18/745 17/741 16/737"); }
    { Vec t={5,5,7,9,2,7}, mach={0,1,2,0,1,1}; int n=6, m=3; Vec L=Loads(n,m,mach,t);
      std::string s=PCMVal(m,L);
      while (PCmaxMoveStep(n,m,t,mach,L,CMAX,BEST)) s+=" "+PCMVal(m,L);
      check("EX_PCM_3 LIST move cmax best", s, "14/441"); }
    { Vec t={5,5,7,9,2,7}, mach={0,1,2,0,1,1}; int n=6, m=3; Vec L=Loads(n,m,mach,t);
      std::string s=PCMVal(m,L);
      while (PCmaxMoveStep(n,m,t,mach,L,SQ,BEST)) s+=" "+PCMVal(m,L);
      check("EX_PCM_3 LIST move sq best", s, "14/441 14/421 12/409"); }
    { Mat d={{0,7,10,12,13,11,7,5,9,4},{7,0,4,7,10,11,7,4,5,3},{10,4,0,4,8,11,7,6,4,6},{12,7,4,0,4,8,6,7,3,9},{13,10,8,4,0,5,6,8,5,11},{11,11,11,8,5,0,4,8,7,11},{7,7,7,6,6,4,0,3,4,6},{5,4,6,7,8,8,3,0,4,3},{9,5,4,3,5,7,4,4,0,6},{4,3,6,9,11,11,6,3,6,0}}; int n=10; long long seed=1; std::vector<int> res;
      for (int s=0;s<100;++s) { Vec pi=RandomTour(n,seed); LocalSearch([&]() { return OrOptStep(n,d,pi,FIRST); }); res.push_back(TourLength(n,d,pi)); }
      check("TSP10 100 starts or first", Hist(res), "39:72 43:26 44:2"); }
    { Mat d={{0,7,10,12,13,11,7,5,9,4},{7,0,4,7,10,11,7,4,5,3},{10,4,0,4,8,11,7,6,4,6},{12,7,4,0,4,8,6,7,3,9},{13,10,8,4,0,5,6,8,5,11},{11,11,11,8,5,0,4,8,7,11},{7,7,7,6,6,4,0,3,4,6},{5,4,6,7,8,8,3,0,4,3},{9,5,4,3,5,7,4,4,0,6},{4,3,6,9,11,11,6,3,6,0}}; int n=10; long long seed=1; std::vector<int> res;
      for (int s=0;s<100;++s) { Vec pi=RandomTour(n,seed); LocalSearch([&]() { return OrOptStep(n,d,pi,BEST); }); res.push_back(TourLength(n,d,pi)); }
      check("TSP10 100 starts or best", Hist(res), "39:77 43:22 44:1"); }
    { Mat d={{0,7,10,12,13,11,7,5,9,4},{7,0,4,7,10,11,7,4,5,3},{10,4,0,4,8,11,7,6,4,6},{12,7,4,0,4,8,6,7,3,9},{13,10,8,4,0,5,6,8,5,11},{11,11,11,8,5,0,4,8,7,11},{7,7,7,6,6,4,0,3,4,6},{5,4,6,7,8,8,3,0,4,3},{9,5,4,3,5,7,4,4,0,6},{4,3,6,9,11,11,6,3,6,0}}; int n=10; long long seed=1; std::vector<int> res;
      for (int s=0;s<100;++s) { Vec pi=RandomTour(n,seed); VND({ [&]() { return TwoOptStep(n,d,pi,FIRST); }, [&]() { return OrOptStep(n,d,pi,FIRST); } }); res.push_back(TourLength(n,d,pi)); }
      check("TSP10 100 starts vnd first", Hist(res), "39:100"); }
    { Mat d={{0,7,10,12,13,11,7,5,9,4},{7,0,4,7,10,11,7,4,5,3},{10,4,0,4,8,11,7,6,4,6},{12,7,4,0,4,8,6,7,3,9},{13,10,8,4,0,5,6,8,5,11},{11,11,11,8,5,0,4,8,7,11},{7,7,7,6,6,4,0,3,4,6},{5,4,6,7,8,8,3,0,4,3},{9,5,4,3,5,7,4,4,0,6},{4,3,6,9,11,11,6,3,6,0}}; int n=10; long long seed=1; std::vector<int> res;
      for (int s=0;s<100;++s) { Vec pi=RandomTour(n,seed); VND({ [&]() { return TwoOptStep(n,d,pi,BEST); }, [&]() { return OrOptStep(n,d,pi,BEST); } }); res.push_back(TourLength(n,d,pi)); }
      check("TSP10 100 starts vnd best", Hist(res), "39:100"); }
    { Mat d={{0,7,10,12,13,11,7,5,9,4},{7,0,4,7,10,11,7,4,5,3},{10,4,0,4,8,11,7,6,4,6},{12,7,4,0,4,8,6,7,3,9},{13,10,8,4,0,5,6,8,5,11},{11,11,11,8,5,0,4,8,7,11},{7,7,7,6,6,4,0,3,4,6},{5,4,6,7,8,8,3,0,4,3},{9,5,4,3,5,7,4,4,0,6},{4,3,6,9,11,11,6,3,6,0}}; int n=(int)d.size(); long long seed=1; std::set<Vec> S;
      for (int s=0;s<100;++s) { Vec pi=RandomTour(n,seed); LocalSearch([&]() { return TwoOptStep(n,d,pi,FIRST); }); S.insert(Canon(pi)); }
      check("TSP10 distinct 2-opt local optima first", std::to_string(S.size()), "2"); }
    { Mat d={{0,7,10,12,13,11,7,5,9,4},{7,0,4,7,10,11,7,4,5,3},{10,4,0,4,8,11,7,6,4,6},{12,7,4,0,4,8,6,7,3,9},{13,10,8,4,0,5,6,8,5,11},{11,11,11,8,5,0,4,8,7,11},{7,7,7,6,6,4,0,3,4,6},{5,4,6,7,8,8,3,0,4,3},{9,5,4,3,5,7,4,4,0,6},{4,3,6,9,11,11,6,3,6,0}}; int n=(int)d.size(); long long seed=1; std::set<Vec> S;
      for (int s=0;s<100;++s) { Vec pi=RandomTour(n,seed); LocalSearch([&]() { return TwoOptStep(n,d,pi,BEST); }); S.insert(Canon(pi)); }
      check("TSP10 distinct 2-opt local optima best", std::to_string(S.size()), "3"); }
    { std::vector<std::pair<int,int>> pts; long long s0=12345; for (int k=0;k<30;++k) { int a=RandomInt(s0,101); int b=RandomInt(s0,101); pts.push_back({a,b}); } Mat d=EuclideanDistances(pts); int n=(int)d.size(); long long seed=1; std::set<Vec> S;
      for (int s=0;s<100;++s) { Vec pi=RandomTour(n,seed); LocalSearch([&]() { return TwoOptStep(n,d,pi,FIRST); }); S.insert(Canon(pi)); }
      check("TSP30 distinct 2-opt local optima first", std::to_string(S.size()), "59"); }
    { std::vector<std::pair<int,int>> pts; long long s0=12345; for (int k=0;k<30;++k) { int a=RandomInt(s0,101); int b=RandomInt(s0,101); pts.push_back({a,b}); } Mat d=EuclideanDistances(pts); int n=(int)d.size(); long long seed=1; std::set<Vec> S;
      for (int s=0;s<100;++s) { Vec pi=RandomTour(n,seed); LocalSearch([&]() { return TwoOptStep(n,d,pi,BEST); }); S.insert(Canon(pi)); }
      check("TSP30 distinct 2-opt local optima best", std::to_string(S.size()), "54"); }
    { std::vector<std::pair<int,int>> pts; long long s0=12345; for (int k=0;k<30;++k) { int a=RandomInt(s0,101); int b=RandomInt(s0,101); pts.push_back({a,b}); } Mat d=EuclideanDistances(pts);
      long long seed=1; int tot=0, mn=1000000, mx=0; for (int s=0;s<100;++s) { Vec pi=RandomTour(30,seed); int L=TourLength(30,d,pi); tot+=L; if (L<mn) mn=L; if (L>mx) mx=L; }
      check("TSP30 random tours: sum min max", std::to_string(tot)+" "+std::to_string(mn)+" "+std::to_string(mx), "176873 1364 2080"); }

    printf("\n%d checks, %d errors\n", checks, errors);
    return errors == 0 ? 0 : 1;
}

Writing src/Main.cpp


In [29]:
!g++ -std=c++17 -O2 -o LOCAL_SEARCH src/*.cpp && ./LOCAL_SEARCH

KP ref: GR                               OK
KP ref: greedy by weight                 OK
BPP ref: FF                              OK
BPP ref: NF                              OK
BPP ref: FFD                             OK
PCmax ref: LPT                           OK
TSP ref: NN from city 1                  OK
TSP10: NN from city 1                    OK
TSP10: distances                         OK
KP ref from (0,0,1,1) first              OK
KP ref from (0,0,1,1) best               OK
KP ref penalty a=1/1 first               OK
KP ref penalty a=1/1 best                OK
KP ref penalty a=2/1 first               OK
KP ref penalty a=2/1 best                OK
KP ref penalty a=9/4 first               OK
KP ref penalty a=9/4 best                OK
KP ref penalty a=3/1 first               OK
KP ref penalty a=3/1 best                OK
BPP ref FF move nb first                 OK
BPP ref FF move nb best                  OK
BPP ref FF move sq first                 OK
BPP ref FF move sq best         

## Try it yourself

Change the data in the cell below and run it again.

In [30]:
%%writefile try.cpp
#include <cstdio>
#include <string>
#include "src/LocalSearch.h"

// try it yourself: local search for the TSP and for P||Cmax on a small instance

static std::string Show(const Vec& v)
{
    std::string s;
    for (size_t k = 0; k < v.size(); ++k) s += (k ? " " : "") + std::to_string(v[k]);
    return s;
}

int main()
{
    // change the data below, compile and run again
    std::vector<std::pair<int,int>> points = {{0,0},{9,1},{2,7},{8,8},{4,3},{1,4},{7,4},{5,9}};   // TSP: coordinates of the cities
    Vec t = {7, 6, 5, 5, 4, 4, 3, 2};          // P||Cmax: processing times of the jobs
    int m = 3;                                 // P||Cmax: number of machines
    int rule = BEST;                           // FIRST or BEST

    int n = (int)points.size();
    Mat d = EuclideanDistances(points);
    long long seed = 1;
    Vec pi = RandomTour(n, seed);
    printf("TSP random tour: %s  length %d\n", Show(pi).c_str(), TourLength(n, d, pi));
    int moves = LocalSearch([&]() { return OrOptStep(n, d, pi, rule); });
    printf("after Or-opt (%d moves): %s  length %d\n", moves, Show(pi).c_str(), TourLength(n, d, pi));
    moves = VND({ [&]() { return TwoOptStep(n, d, pi, rule); }, [&]() { return OrOptStep(n, d, pi, rule); } });
    printf("after VND 2-opt + Or-opt (%d moves): %s  length %d\n", moves, Show(pi).c_str(), TourLength(n, d, pi));
    Vec best;
    int length = MultiStartTwoOpt(n, d, 20, 1, rule, best);
    printf("multi-start 2-opt, 20 starts: %s  length %d\n", Show(best).c_str(), length);

    int nj = (int)t.size();
    Vec mach = RandomAssignment(nj, m, seed);
    Vec L = Loads(nj, m, mach, t);
    printf("P||Cmax random assignment: %s  loads %s\n", Show(mach).c_str(), Show(L).c_str());
    moves = VND({ [&]() { return PCmaxMoveStep(nj, m, t, mach, L, SQ, rule); },
                  [&]() { return PCmaxSwapStep(nj, m, t, mach, L, SQ, rule); } });
    printf("after VND move + swap (%d moves): %s  loads %s\n", moves, Show(mach).c_str(), Show(L).c_str());
    mach = LPT(nj, m, t);
    printf("LPT, for comparison: %s  loads %s\n", Show(mach).c_str(), Show(Loads(nj, m, mach, t)).c_str());
    return 0;
}

Writing try.cpp


In [31]:
!g++ -std=c++17 -O2 -o try try.cpp $(ls src/*.cpp | grep -v Main.cpp) && ./try

TSP random tour: 0 3 6 5 2 4 7 1  length 52
after Or-opt (2 moves): 0 5 2 7 3 6 1 4  length 32
after VND 2-opt + Or-opt (0 moves): 0 5 2 7 3 6 1 4  length 32
multi-start 2-opt, 20 starts: 0 4 1 6 3 7 2 5  length 32
P||Cmax random assignment: 0 2 2 2 1 1 2 0  loads 9 8 19
after VND move + swap (4 moves): 0 1 0 2 2 1 2 1  loads 12 12 12
LPT, for comparison: 0 1 2 2 1 0 1 2  loads 11 13 12


---

Teaching material by [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza University of Rome.
Text, figures and data [CC BY 4.0](https://github.com/fabiofurini/combinatorial-optimization/blob/main/LICENSE), code [MIT](https://github.com/fabiofurini/combinatorial-optimization/blob/main/LICENSE-CODE).